# 🛢️ Naija-Petro — Petroleum Engineering Corpus Builder

**Notebook 1 of 5 · Naija-Petro project**

End-to-end pipeline that builds the fine-tuning corpus: **scrape → clean → generate (NVIDIA Data Designer) → export**. The output is 20,000+ instruction–response pairs used to fine-tune the Naija-Petro models ([`Shinzmann/naija-petro-8b`](https://huggingface.co/Shinzmann/naija-petro-8b) and [`Shinzmann/naija-petro`](https://huggingface.co/Shinzmann/naija-petro)).

> Inspired by *Adapting Large Language Models for Oil and Gas Applications Through Domain-Specific Fine-Tuning*. Synthetic generation uses NVIDIA NeMo Data Designer (`pip install data-designer`).

### Pipeline overview

| Phase | Description | Tool |
|-------|-------------|------|
| 0 | Setup & infrastructure | Python, requests |
| 1 | Query-list preparation | 60+ arXiv, 100+ academic, 30+ DOE queries |
| 2 | Scrape 25+ sources | arXiv, Semantic Scholar, OpenAlex, PetroWiki, SLB, … |
| 3 | Consolidate seed data | Deduplicate, chunk, export as a Parquet seed |
| 4 | **NVIDIA Data Designer** | Generate instruction–response pairs with an LLM |
| | → Pipeline 1: Knowledge generation | Sampler-driven diverse QA |
| | → Pipeline 2: Seed-grounded QA | Corpus-contextualised QA |
| | → Pipeline 3: Quality scoring | LLM-as-Judge (accuracy, completeness, usefulness) |
| 5 | Export | Alpaca JSONL, ShareGPT JSONL, Parquet |

---

## PHASE 0: Setup & Core Infrastructure

In [ ]:
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
!pip install requests beautifulsoup4 pandas tqdm lxml feedparser \
    trafilatura fake-useragent openpyxl jsonlines pyarrow \
    retry tenacity newspaper3k -q
!pip install "pyarrow>=19.0.1,<20" --break-system-packages -q
print("✅ Dependencies installed.")

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 44.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os
import re
import json
import time
import random
import hashlib
import logging
import requests
import feedparser
import jsonlines
import pandas as pd
from pathlib import Path
from datetime import datetime
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from urllib.parse import urljoin, quote_plus
from collections import Counter

try:
    import trafilatura
except ImportError:
    trafilatura = None
    print("⚠️ trafilatura not available, using BS4 fallback")

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

print("✅ All imports loaded.")

✅ All imports loaded.


In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE & CONFIGURATION
# ============================================================
from google.colab import userdata
EIA_API_KEY= userdata.get('EIA_API_KEY')        # https://www.eia.gov/opendata/register.php
POLITE_EMAIL = userdata.get('POLITE_EMAIL')         # Your email for OpenAlex/Crossref polite pool (10x rate)

CORE_API_KEY = userdata.get('CORE_API_KEY')        # https://core.ac.uk/services/api (free, 10k/month)

# --- Output: Google Drive (persists across sessions) ---
DRIVE_BASE = Path("/content/drive/MyDrive/petroleum_corpus")

OUTPUT_DIR = DRIVE_BASE
RAW_DIR = OUTPUT_DIR / "raw"
PROCESSED_DIR = OUTPUT_DIR / "processed"
METADATA_DIR = OUTPUT_DIR / "metadata"
AUGMENTED_DIR = OUTPUT_DIR / "augmented"

for d in [RAW_DIR, PROCESSED_DIR, METADATA_DIR, AUGMENTED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Scraping controls ---
REQUEST_DELAY = (1.5, 3.0)        # Polite random delay between requests
MAX_ITEMS_PER_SOURCE = 3000       # Upper bound per source
MAX_ITEMS_PER_QUERY = 300         # Per individual query within a source

print(f"✅ Google Drive mounted.")
print(f"✅ All outputs will be saved to: {OUTPUT_DIR}")
print(f"   This persists across Colab sessions!")

✅ Google Drive mounted.
✅ All outputs will be saved to: /content/drive/MyDrive/petroleum_corpus
   This persists across Colab sessions!


In [ ]:
# ============================================================
# CORE UTILITIES
# ============================================================

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}

session = requests.Session()
retry_strategy = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)
session.headers.update(HEADERS)


def polite_delay(multiplier=1.0):
    time.sleep(random.uniform(*REQUEST_DELAY) * multiplier)


def safe_request(url, params=None, headers=None, timeout=30):
    for attempt in range(3):
        try:
            resp = session.get(url, params=params, headers=headers, timeout=timeout)
            if resp.status_code == 429:
                wait = 2 ** (attempt + 2)
                logger.warning(f"Rate limited ({url}). Waiting {wait}s...")
                time.sleep(wait)
                continue
            if resp.status_code == 403:
                logger.warning(f"403 Forbidden: {url}")
                return None
            resp.raise_for_status()
            return resp
        except requests.exceptions.RequestException as e:
            logger.error(f"Request failed (attempt {attempt+1}/3): {e}")
            if attempt < 2:
                time.sleep(2 ** attempt)
    return None


def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    return text


def generate_id(text):
    return hashlib.md5(text.encode('utf-8')).hexdigest()[:12]


def save_records(records, filename, subdir="raw"):
    dirmap = {"raw": RAW_DIR, "processed": PROCESSED_DIR, "augmented": AUGMENTED_DIR}
    outpath = dirmap.get(subdir, RAW_DIR) / filename
    with jsonlines.open(outpath, mode='w') as writer:
        for rec in records:
            writer.write(rec)
    logger.info(f"💾 Saved {len(records)} records → {outpath}")
    return outpath


def extract_text_from_html(html_content, url=None):
    if trafilatura:
        text = trafilatura.extract(html_content, include_comments=False, include_tables=True)
        if text:
            return clean_text(text)
    soup = BeautifulSoup(html_content, 'lxml')
    for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
        tag.decompose()
    return clean_text(soup.get_text(separator=' '))


# Global tracker
SCRAPE_LOG = {}  # source_name -> record_count

def log_source(name, count):
    SCRAPE_LOG[name] = count
    total = sum(SCRAPE_LOG.values())
    print(f"\n{'='*50}")
    print(f"📊 Running total: {total:,} records from {len(SCRAPE_LOG)} sources")
    print(f"{'='*50}")


print("✅ Utilities ready.")

✅ Utilities ready.


---
## PHASE 1: MEGA QUERY LISTS

3x expanded queries across every petroleum engineering subdomain.

In [ ]:
# ============================================================
# EXPANDED QUERY LISTS — 3X COVERAGE
# ============================================================

# --- arXiv queries (ML + petroleum intersection) ---
ARXIV_QUERIES = [
    # Core intersections
    'all:"petroleum engineering" AND (all:"machine learning" OR all:"deep learning")',
    'all:"reservoir simulation" AND all:"neural network"',
    'all:"oil and gas" AND all:"artificial intelligence"',
    'all:"well log" AND all:"machine learning"',
    'all:"production optimization" AND all:"petroleum"',
    'all:"enhanced oil recovery" AND (all:"data-driven" OR all:"machine learning")',
    'all:"drilling optimization" AND all:"machine learning"',
    'all:"reservoir characterization" AND all:"deep learning"',
    'all:"hydraulic fracturing" AND all:"data-driven"',
    'all:"flow assurance" AND (all:"prediction" OR all:"machine learning")',
    'all:"seismic interpretation" AND all:"deep learning"',
    'all:"PVT" AND all:"machine learning"',
    'all:"pore pressure prediction" AND all:"neural network"',
    'all:"digital twin" AND all:"oil and gas"',
    'all:"anomaly detection" AND all:"oil and gas"',
    'all:"natural language processing" AND all:"petroleum"',
    'all:"large language model" AND (all:"oil" OR all:"petroleum" OR all:"energy")',
    'all:"multiphase flow" AND all:"machine learning"',
    'all:"subsurface" AND all:"deep learning"',
    'all:"petrophysics" AND all:"machine learning"',
    # Expanded
    'all:"formation evaluation" AND all:"neural network"',
    'all:"well testing" AND all:"machine learning"',
    'all:"production forecasting" AND all:"deep learning"',
    'all:"decline curve" AND all:"machine learning"',
    'all:"rock physics" AND all:"deep learning"',
    'all:"facies classification" AND all:"deep learning"',
    'all:"history matching" AND all:"machine learning"',
    'all:"well placement" AND all:"optimization"',
    'all:"pipeline leak detection" AND all:"machine learning"',
    'all:"corrosion prediction" AND all:"neural network"',
    'all:"sand production" AND all:"prediction"',
    'all:"gas hydrate" AND all:"machine learning"',
    'all:"wax deposition" AND all:"prediction model"',
    'all:"geomechanics" AND all:"machine learning"',
    'all:"wellbore stability" AND all:"prediction"',
    'all:"carbon capture" AND all:"reservoir" AND all:"machine learning"',
    'all:"shale gas" AND all:"machine learning"',
    'all:"tight oil" AND all:"production prediction"',
    'all:"drilling rate of penetration" AND all:"prediction"',
    'all:"stuck pipe" AND all:"prediction"',
    'all:"lost circulation" AND all:"machine learning"',
    'all:"kick detection" AND all:"real-time"',
    'all:"ESP" AND all:"failure prediction"',
    'all:"gas lift" AND all:"optimization" AND all:"machine learning"',
    'all:"water breakthrough" AND all:"prediction"',
    'all:"EOR screening" AND all:"machine learning"',
    'all:"fracture network" AND all:"deep learning"',
    'all:"core analysis" AND all:"image recognition"',
    'all:"thin section" AND all:"deep learning"',
    'all:"seismic inversion" AND all:"deep learning"',
    'all:"petroleum" AND all:"transformer" AND all:"model"',
    'all:"oil spill" AND all:"detection" AND all:"deep learning"',
    'all:"smart well" AND all:"optimization"',
    'all:"proxy model" AND all:"reservoir simulation"',
    'all:"surrogate model" AND all:"petroleum"',
    'all:"physics-informed" AND all:"neural network" AND all:"reservoir"',
    'all:"graph neural network" AND all:"reservoir"',
    'all:"reinforcement learning" AND all:"oil" AND all:"production"',
    'all:"generative adversarial" AND all:"reservoir"',
    'all:"transfer learning" AND all:"well log"',
    'all:"autoencoder" AND all:"seismic"',
]

# --- Academic paper queries (Semantic Scholar, OpenAlex, Crossref) ---
ACADEMIC_QUERIES = [
    # Reservoir engineering
    "petroleum reservoir engineering",
    "reservoir simulation history matching",
    "reservoir characterization geostatistics",
    "material balance equation reservoir",
    "decline curve analysis production forecasting",
    "reservoir fluid properties PVT",
    "relative permeability measurement",
    "capillary pressure wettability",
    "waterflood performance prediction",
    "gas reservoir deliverability",
    "gas condensate reservoir management",
    "naturally fractured reservoir simulation",
    "reservoir pressure maintenance",
    "inflow performance relationship IPR",
    "well deliverability nodal analysis",
    # Production engineering
    "oil gas production optimization",
    "artificial lift systems design",
    "electric submersible pump ESP design",
    "gas lift optimization design",
    "sucker rod pump design",
    "progressive cavity pump application",
    "plunger lift design optimization",
    "production system optimization",
    "well performance analysis",
    "multiphase flow metering",
    "sand management production",
    "water management oil production",
    "scale inhibition oilfield",
    "paraffin wax management",
    "asphaltene management strategy",
    "emulsion treatment petroleum",
    "separator design oil gas",
    # Drilling engineering
    "drilling engineering rate of penetration",
    "directional drilling trajectory",
    "horizontal well drilling",
    "managed pressure drilling",
    "underbalanced drilling",
    "drilling fluid mud design",
    "well control blow out prevention",
    "casing design oil well",
    "cementing oil gas well",
    "drill bit selection optimization",
    "drilling vibration management",
    "wellbore stability analysis",
    "stuck pipe prevention",
    "lost circulation management",
    "coiled tubing operations",
    "slim hole drilling technology",
    # Completion & stimulation
    "well completion design",
    "hydraulic fracturing design optimization",
    "acid stimulation treatment",
    "gravel pack completion",
    "sand control petroleum",
    "perforating design oil well",
    "multistage fracturing horizontal well",
    "proppant selection design",
    "fracture conductivity optimization",
    "refracturing treatment",
    "intelligent completion design",
    "expandable tubular technology",
    # EOR
    "enhanced oil recovery methods",
    "chemical flooding polymer surfactant",
    "thermal recovery SAGD CSS",
    "CO2 flooding miscible",
    "microbial enhanced oil recovery",
    "low salinity waterflooding",
    "nanoparticle enhanced oil recovery",
    "foam flooding EOR",
    "alkaline surfactant polymer ASP",
    "WAG water alternating gas",
    "EOR screening criteria",
    # Formation evaluation
    "formation evaluation petrophysics",
    "well log interpretation",
    "resistivity log interpretation",
    "neutron density log analysis",
    "nuclear magnetic resonance NMR log",
    "acoustic log sonic interpretation",
    "formation testing sampling",
    "mud log analysis",
    "core analysis laboratory",
    "saturation height function",
    # Flow assurance
    "flow assurance deepwater",
    "gas hydrate prediction prevention",
    "wax deposition prediction",
    "asphaltene precipitation model",
    "multiphase flow pipeline",
    "slug flow prediction",
    "pipeline corrosion prediction",
    "erosion prediction petroleum",
    "flow assurance thermal management",
    "chemical injection flow assurance",
    # Offshore & subsea
    "offshore petroleum engineering",
    "FPSO operations production",
    "subsea production system design",
    "deepwater drilling technology",
    "subsea tieback design",
    "riser design analysis",
    "mooring system design",
    "subsea manifold design",
    "umbilical design subsea",
    "subsea processing technology",
    # Process engineering & facilities
    "oil gas process simulation",
    "natural gas processing design",
    "gas dehydration glycol",
    "gas sweetening amine",
    "NGL recovery process",
    "petroleum refining process",
    "produced water treatment",
    "flare system design",
    "compressor station design",
    "heat exchanger fouling petroleum",
    # Safety & integrity
    "process safety management oil gas",
    "well integrity management",
    "pipeline integrity management",
    "risk assessment oil gas",
    "blowout preventer testing",
    "HAZOP petroleum facility",
    "safety instrumented system SIS",
    # Unconventional
    "shale gas production technology",
    "tight oil reservoir engineering",
    "coalbed methane production",
    "oil sands extraction technology",
    "heavy oil recovery method",
    # Digital / AI
    "digital twin oil gas operations",
    "machine learning petroleum engineering",
    "artificial intelligence drilling",
    "data analytics oil gas",
    "IoT sensors oil gas monitoring",
    "real-time optimization petroleum",
    "predictive maintenance oil gas",
    "computer vision oil gas inspection",
    "NLP petroleum technical documents",
    # Emerging
    "carbon capture utilization storage",
    "hydrogen production petroleum",
    "geothermal reservoir engineering",
    "methane emission detection",
    "energy transition oil gas",
]

# --- DOE/OSTI-specific queries ---
DOE_QUERIES = [
    "petroleum engineering reservoir", "oil gas production technology",
    "enhanced oil recovery research", "drilling technology advancement",
    "hydraulic fracturing optimization", "well stimulation treatment",
    "oil gas environmental monitoring", "petroleum refinery technology",
    "carbon capture storage oil gas", "unconventional oil gas shale",
    "offshore oil gas technology", "deepwater petroleum engineering",
    "artificial intelligence oil gas", "digital twin petroleum",
    "machine learning reservoir", "natural gas processing technology",
    "well integrity testing", "pipeline corrosion mitigation",
    "produced water management", "gas hydrate research",
    "subsurface characterization", "petrophysical analysis",
    "geomechanical modeling", "seismic processing interpretation",
    "multiphase flow modeling", "thermal recovery heavy oil",
    "chemical EOR flooding", "microseismic monitoring",
    "wellbore stability geomechanics", "formation damage control",
]

# --- Wikipedia topics ---
WIKI_TOPICS = [
    "petroleum engineering", "reservoir engineering", "drilling engineering",
    "production engineering", "well completion", "well intervention",
    "formation evaluation", "petrophysics", "well logging",
    "enhanced oil recovery", "waterflooding", "gas injection EOR",
    "chemical flooding", "thermal recovery", "polymer flooding",
    "decline curve analysis", "material balance equation",
    "Darcy's law", "reservoir simulation", "history matching reservoir",
    "directional drilling", "horizontal drilling",
    "managed pressure drilling", "underbalanced drilling",
    "drill bit types", "drilling fluid", "well control blowout",
    "casing design oil well", "cementing oil well",
    "hydraulic fracturing", "acid stimulation petroleum", "gravel pack completion",
    "sand control petroleum", "perforating oil well",
    "gas lift", "electric submersible pump", "sucker rod pump",
    "progressive cavity pump", "plunger lift",
    "oil gas separator", "petroleum refining", "natural gas processing",
    "dehydration natural gas", "sweetening natural gas",
    "flow assurance", "gas hydrate pipeline", "wax deposition pipeline",
    "asphaltene deposition", "scale inhibition oilfield",
    "multiphase flow pipeline", "slug flow",
    "offshore drilling", "FPSO floating production",
    "subsea production system", "deepwater drilling",
    "jack-up rig", "semi-submersible platform",
    "multiphase flow meter", "well testing petroleum",
    "pressure transient analysis", "production logging",
    "petroleum geology", "source rock petroleum", "sedimentary basin",
    "seismic survey exploration", "well log correlation",
    "process safety management", "blowout preventer",
    "HAZOP analysis", "flare system petroleum",
    "shale gas", "tight oil", "coalbed methane", "oil sands",
    "API gravity", "porosity", "permeability geology",
    "water cut petroleum", "gas oil ratio", "bubble point pressure",
    "PVT analysis petroleum", "relative permeability",
    "capillary pressure", "wettability",
    # Additional
    "Christmas tree wellhead", "tubing head pressure",
    "bottomhole pressure", "skin factor well",
    "Horner plot", "Bourdet derivative",
    "Buckley Leverett", "fractional flow",
    "Vogel IPR", "Fetkovich method",
    "Archie equation", "Wyllie time average",
    "Gassmann equation", "Hashin Shtrikman bounds",
    "oil well", "gas well", "injection well",
    "workover operations", "fishing operations oil well",
    "well abandonment plugging", "slot recovery",
    "drill string", "bottom hole assembly",
    "measurement while drilling", "logging while drilling",
    "rotary steerable system", "mud motor",
    "petroleum coke", "natural gas liquids",
    "liquefied natural gas LNG", "compressed natural gas CNG",
    "OPEC petroleum", "petroleum reserves classification",
    "SPE PRMS reserves", "proved reserves",
]

print(f"✅ Query lists built:")
print(f"   arXiv queries:      {len(ARXIV_QUERIES)}")
print(f"   Academic queries:   {len(ACADEMIC_QUERIES)}")
print(f"   DOE queries:        {len(DOE_QUERIES)}")
print(f"   Wikipedia topics:   {len(WIKI_TOPICS)}")

✅ Query lists built:
   arXiv queries:      61
   Academic queries:   137
   DOE queries:        30
   Wikipedia topics:   119


---
## Phase 2 — Scrape all sources

Each section below targets one open, citable source. The scrapers are independent and fault-tolerant: a failure in one source never blocks the others, and raw results are written to Drive as they arrive (so a disconnect never loses progress).

### 2.1 — arXiv (AI/ML + Petroleum)

In [ ]:
def scrape_arxiv(queries, max_per_query=200):
    records, seen_ids = [], set()
    base_url = "http://export.arxiv.org/api/query"

    for query in tqdm(queries, desc="arXiv"):
        start, qcount = 0, 0
        while qcount < max_per_query:
            params = {"search_query": query, "start": start, "max_results": min(100, max_per_query - qcount), "sortBy": "relevance"}
            resp = safe_request(base_url, params=params)
            if not resp: break
            feed = feedparser.parse(resp.text)
            if not feed.entries: break
            for entry in feed.entries:
                aid = entry.get('id', '').split('/abs/')[-1]
                if aid in seen_ids: continue
                seen_ids.add(aid)
                title = clean_text(entry.get('title', ''))
                abstract = clean_text(entry.get('summary', ''))
                if not abstract or len(abstract) < 50: continue
                authors = [a.get('name', '') for a in entry.get('authors', [])]
                categories = [t.get('term', '') for t in entry.get('tags', [])]
                records.append({
                    "id": generate_id(aid), "source": "arxiv", "arxiv_id": aid,
                    "title": title, "abstract": abstract, "authors": authors,
                    "categories": categories, "published": entry.get('published', ''),
                    "url": entry.get('id', ''), "content_type": "research_paper",
                    "text": f"Title: {title}\n\nAbstract: {abstract}",
                    "scraped_at": datetime.now().isoformat()
                })
                qcount += 1
            start += 100
            polite_delay()
            if len(feed.entries) < 100: break
    logger.info(f"arXiv: {len(records)} papers")
    return records

arxiv_records = scrape_arxiv(ARXIV_QUERIES, max_per_query=150)
save_records(arxiv_records, "01_arxiv.jsonl")
log_source("arxiv", len(arxiv_records))

arXiv:   0%|          | 0/61 [00:00<?, ?it/s]


📊 Running total: 739 records from 1 sources


### 2.2 — Semantic Scholar

In [ ]:
os.environ['S2_API_KEY'] = userdata.get('S2_API_KEY')

def scrape_semantic_scholar(queries, max_per_query=200):
    """
    Semantic Scholar API — STRICT rate limits: 1 req/sec unauthenticated.
    Uses longer delays and smaller batches to avoid 429 cascades.
    Get a free API key at https://www.semanticscholar.org/product/api#api-key
    for 10 req/sec (highly recommended).
    """
    records, seen = [], set()
    base_url = "https://api.semanticscholar.org/graph/v1/paper/search"
    fields = "paperId,title,abstract,authors,year,citationCount,fieldsOfStudy,url,publicationDate"

    # Use API key if available (10x rate limit increase)
    s2_headers = {}
    S2_API_KEY = os.environ.get('S2_API_KEY', '')
    if S2_API_KEY:
        s2_headers['x-api-key'] = S2_API_KEY
        s2_delay = 0.15  # 10 req/sec with key
        batch_size = 100
        print("✅ Using Semantic Scholar API key (10 req/sec)")
    else:
        s2_delay = 1.2  # Stay safely under 1 req/sec
        batch_size = 50  # Smaller batches = fewer retries needed
        print("⚠️  No S2 API key — using slow mode (1 req/sec).")
        print("   Get a free key: https://www.semanticscholar.org/product/api#api-key")
        print("   Set it: os.environ['S2_API_KEY'] = userdata.get('S2_API_KEY')")

    failed_streak = 0  # Track consecutive failures

    for qi, query in enumerate(tqdm(queries, desc="Semantic Scholar")):
        offset, qcount = 0, 0

        # If we've had too many consecutive failures, back off hard
        if failed_streak >= 5:
            logger.warning("Too many consecutive 429s. Pausing 60s...")
            time.sleep(60)
            failed_streak = 0

        while qcount < max_per_query:
            params = {
                "query": query,
                "offset": offset,
                "limit": min(batch_size, max_per_query - qcount),
                "fields": fields,
            }

            # Manual request with proper 429 handling
            resp = None
            for attempt in range(3):
                try:
                    resp = requests.get(
                        base_url, params=params,
                        headers={**HEADERS, **s2_headers},
                        timeout=30
                    )
                    if resp.status_code == 429:
                        # Exponential backoff: 5s, 15s, 45s
                        wait = 5 * (3 ** attempt)
                        logger.warning(f"S2 rate limited. Waiting {wait}s (attempt {attempt+1}/3)")
                        time.sleep(wait)
                        failed_streak += 1
                        continue
                    resp.raise_for_status()
                    failed_streak = 0  # Reset on success
                    break
                except requests.exceptions.RequestException as e:
                    if attempt < 2:
                        time.sleep(5 * (attempt + 1))
                    else:
                        logger.error(f"S2 request failed after 3 attempts: {e}")
                        resp = None

            if not resp or resp.status_code != 200:
                break

            try:
                data = resp.json()
            except:
                break
            papers = data.get('data', [])
            if not papers:
                break

            for p in papers:
                pid = p.get('paperId', '')
                if pid in seen or not p.get('abstract'):
                    continue
                seen.add(pid)
                title = clean_text(p.get('title', ''))
                abstract = clean_text(p.get('abstract', ''))
                if len(abstract) < 50:
                    continue
                authors = [a.get('name', '') for a in (p.get('authors') or [])]
                records.append({
                    "id": generate_id(pid), "source": "semantic_scholar",
                    "paper_id": pid, "title": title, "abstract": abstract,
                    "authors": authors, "year": p.get('year'),
                    "citations": p.get('citationCount', 0),
                    "fields": p.get('fieldsOfStudy', []),
                    "content_type": "research_paper",
                    "text": f"Title: {title}\n\nAbstract: {abstract}",
                    "scraped_at": datetime.now().isoformat()
                })
                qcount += 1

            offset += batch_size
            time.sleep(s2_delay)  # Strict per-request delay
            if len(papers) < batch_size:
                break

        # Inter-query delay (extra breathing room)
        time.sleep(s2_delay * 2)

        # Progress update every 10 queries
        if (qi + 1) % 10 == 0:
            logger.info(f"S2 progress: {qi+1}/{len(queries)} queries, {len(records)} papers so far")

    logger.info(f"Semantic Scholar: {len(records)} papers")
    return records

# NOTE: Semantic Scholar is SLOW without an API key (~1 req/sec).
# Reduce max_per_query to speed up, or get a free API key.
ss_records = scrape_semantic_scholar(ACADEMIC_QUERIES, max_per_query=100)
save_records(ss_records, "02_semantic_scholar.jsonl")
log_source("semantic_scholar", len(ss_records))

✅ Using Semantic Scholar API key (10 req/sec)


Semantic Scholar:   0%|          | 0/137 [00:00<?, ?it/s]


📊 Running total: 10,082 records from 1 sources


### 2.3 — OpenAlex

In [ ]:
def scrape_openalex(queries, max_per_query=200):
    records, seen = [], set()
    base_url = "https://api.openalex.org/works"

    for query in tqdm(queries, desc="OpenAlex"):
        page, qcount = 1, 0
        while qcount < max_per_query:
            params = {"search": query, "per_page": 100, "page": page,
                      "filter": "has_abstract:true,type:article", "sort": "relevance_score:desc",
                      "select": "id,title,abstract_inverted_index,authorships,publication_year,cited_by_count,concepts,doi"}
            if POLITE_EMAIL: params["mailto"] = POLITE_EMAIL
            resp = safe_request(base_url, params=params)
            if not resp: break
            try: data = resp.json()
            except: break
            works = data.get('results', [])
            if not works: break
            for w in works:
                wid = w.get('id', '')
                if wid in seen: continue
                seen.add(wid)
                aii = w.get('abstract_inverted_index', {})
                if not aii: continue
                wp = []
                for word, positions in aii.items():
                    for pos in positions: wp.append((pos, word))
                wp.sort()
                abstract = clean_text(' '.join([x[1] for x in wp]))
                if len(abstract) < 50: continue
                title = clean_text(w.get('title', ''))
                authors = [a.get('author',{}).get('display_name','') for a in w.get('authorships',[])]
                records.append({
                    "id": generate_id(wid), "source": "openalex", "openalex_id": wid,
                    "title": title, "abstract": abstract, "authors": authors,
                    "year": w.get('publication_year'), "citations": w.get('cited_by_count',0),
                    "doi": w.get('doi',''), "content_type": "research_paper",
                    "text": f"Title: {title}\n\nAbstract: {abstract}",
                    "scraped_at": datetime.now().isoformat()
                })
                qcount += 1
            page += 1
            polite_delay()
            if len(works) < 100: break
    logger.info(f"OpenAlex: {len(records)} papers")
    return records

oa_records = scrape_openalex(ACADEMIC_QUERIES, max_per_query=100)
save_records(oa_records, "03_openalex.jsonl")
log_source("openalex", len(oa_records))

OpenAlex:   0%|          | 0/137 [00:00<?, ?it/s]


📊 Running total: 24,731 records from 1 sources


### 2.4 — Crossref (Petroleum Journals)

In [ ]:
def scrape_crossref(queries, max_per_query=200):
    records, seen = [], set()
    base_url = "https://api.crossref.org/works"

    for query in tqdm(queries, desc="Crossref"):
        offset, qcount = 0, 0
        while qcount < max_per_query:
            params = {"query": query, "rows": min(100, max_per_query-qcount), "offset": offset,
                      "filter": "type:journal-article,has-abstract:true", "sort": "relevance"}
            if POLITE_EMAIL: params["mailto"] = POLITE_EMAIL
            resp = safe_request(base_url, params=params)
            if not resp: break
            try: data = resp.json()
            except: break
            items = data.get('message',{}).get('items',[])
            if not items: break
            for item in items:
                doi = item.get('DOI','')
                if doi in seen: continue
                seen.add(doi)
                abstract = re.sub(r'<[^>]+>', '', item.get('abstract',''))
                abstract = clean_text(abstract)
                if len(abstract) < 50: continue
                tlist = item.get('title',[])
                title = clean_text(tlist[0] if tlist else '')
                authors = [f"{a.get('given','')} {a.get('family','')}".strip() for a in item.get('author',[])]
                journal = item.get('container-title',[''])
                journal = journal[0] if journal else ''
                pd_raw = item.get('published-print',{}).get('date-parts',[[None]])
                year = pd_raw[0][0] if pd_raw and pd_raw[0] else None
                records.append({
                    "id": generate_id(doi), "source": "crossref", "doi": doi,
                    "title": title, "abstract": abstract, "authors": authors,
                    "journal": journal, "year": year,
                    "citations": item.get('is-referenced-by-count',0),
                    "content_type": "research_paper",
                    "text": f"Title: {title}\nJournal: {journal}\n\nAbstract: {abstract}",
                    "scraped_at": datetime.now().isoformat()
                })
                qcount += 1
            offset += 100
            polite_delay()
            if len(items) < 100: break
    logger.info(f"Crossref: {len(records)} papers")
    return records

cr_records = scrape_crossref(ACADEMIC_QUERIES, max_per_query=80)
save_records(cr_records, "04_crossref.jsonl")
log_source("crossref", len(cr_records))

Crossref:   0%|          | 0/137 [00:00<?, ?it/s]


📊 Running total: 34,668 records from 2 sources


### 2.5 — DOE / OSTI Technical Reports

In [ ]:
def scrape_doe_osti(queries, max_per_query=200):
    records, seen = [], set()
    base_url = "https://www.osti.gov/api/v1/records"
    for query in tqdm(queries, desc="DOE OSTI"):
        page, qcount = 0, 0
        while qcount < max_per_query:
            params = {"q": query, "rows": 100, "page": page}
            hdr = {**HEADERS, "Accept": "application/json"}
            try:
                resp = session.get(base_url, params=params, headers=hdr, timeout=30)
                resp.raise_for_status()
                data = resp.json()
            except: break
            if not data: break
            for item in data:
                oid = str(item.get('osti_id',''))
                if oid in seen: continue
                seen.add(oid)
                title = clean_text(item.get('title',''))
                desc = clean_text(item.get('description',''))
                if len(desc) < 50: continue
                records.append({
                    "id": generate_id(oid), "source": "doe_osti", "osti_id": oid,
                    "title": title, "abstract": desc,
                    "year": item.get('publication_date','')[:4] if item.get('publication_date') else None,
                    "org": clean_text(item.get('research_org','')),
                    "content_type": "technical_report",
                    "text": f"Title: {title}\nOrganization: {clean_text(item.get('research_org',''))}\n\nAbstract: {desc}",
                    "scraped_at": datetime.now().isoformat()
                })
                qcount += 1
            page += 1; polite_delay()
            if len(data) < 100: break
    logger.info(f"DOE OSTI: {len(records)} reports")
    return records

doe_records = scrape_doe_osti(DOE_QUERIES, max_per_query=150)
save_records(doe_records, "05_doe_osti.jsonl")
log_source("doe_osti", len(doe_records))

DOE OSTI:   0%|          | 0/30 [00:00<?, ?it/s]


📊 Running total: 40,201 records from 3 sources


### 2.6 — CORE (Open Access Full Texts)

In [ ]:
from google.colab import userdata
CORE_API_KEY= userdata.get('CORE_API_KEY')

def scrape_core(queries, max_per_query=150, api_key=None):
    """
    CORE aggregates open-access research. Free API: https://core.ac.uk/services/api
    Returns full-text for many papers — huge advantage for corpus building.
    """
    if not api_key:
        print("⚠️ No CORE API key. Get one free at https://core.ac.uk/services/api")
        print("   Skipping CORE. Set CORE_API_KEY and re-run.")
        return []

    records, seen = [], set()
    base_url = "https://api.core.ac.uk/v3/search/works"
    headers_core = {"Authorization": f"Bearer {api_key}"}

    for query in tqdm(queries, desc="CORE"):
        offset, qcount = 0, 0
        while qcount < max_per_query:
            params = {"q": query, "limit": 100, "offset": offset}
            try:
                resp = session.get(base_url, params=params, headers=headers_core, timeout=30)
                resp.raise_for_status()
                data = resp.json()
            except Exception as e:
                logger.error(f"CORE error: {e}")
                break

            results = data.get('results', [])
            if not results: break

            for item in results:
                cid = str(item.get('id', ''))
                if cid in seen: continue
                seen.add(cid)

                title = clean_text(item.get('title', ''))
                abstract = clean_text(item.get('abstract', ''))
                fulltext = clean_text(item.get('fullText', ''))

                # Prefer fulltext, fall back to abstract
                text = fulltext if fulltext and len(fulltext) > 200 else abstract
                if not text or len(text) < 100: continue

                # Truncate very long fulltexts to 5000 words
                words = text.split()
                if len(words) > 5000:
                    text = ' '.join(words[:5000])

                authors = [a.get('name','') for a in item.get('authors', [])]

                records.append({
                    "id": generate_id(cid), "source": "core", "core_id": cid,
                    "title": title, "abstract": abstract[:500],
                    "has_fulltext": bool(fulltext and len(fulltext) > 200),
                    "authors": authors, "year": item.get('yearPublished'),
                    "content_type": "research_paper",
                    "text": f"Title: {title}\n\n{text}",
                    "scraped_at": datetime.now().isoformat()
                })
                qcount += 1

            offset += 100
            polite_delay()
            if len(results) < 100: break

    logger.info(f"CORE: {len(records)} papers ({sum(1 for r in records if r.get('has_fulltext'))} with full text)")
    return records

CORE_QUERIES = [
    "petroleum engineering", "reservoir simulation", "enhanced oil recovery",
    "drilling optimization", "well logging interpretation", "hydraulic fracturing",
    "flow assurance pipeline", "production optimization oil gas",
    "multiphase flow petroleum", "formation evaluation petrophysics",
    "subsea production engineering", "artificial lift design",
    "machine learning petroleum", "digital oilfield",
    "carbon capture storage reservoir", "well testing analysis",
]

core_records = scrape_core(CORE_QUERIES, max_per_query=120, api_key=CORE_API_KEY if CORE_API_KEY else None)
if core_records:
    save_records(core_records, "06_core.jsonl")
log_source("core", len(core_records))

CORE:   0%|          | 0/16 [00:00<?, ?it/s]

ERROR:__main__:CORE error: HTTPSConnectionPool(host='api.core.ac.uk', port=443): Max retries exceeded with url: /v3/search/works/?q=drilling+optimization&limit=100&offset=0 (Caused by ResponseError('too many 429 error responses'))
ERROR:__main__:CORE error: HTTPSConnectionPool(host='api.core.ac.uk', port=443): Max retries exceeded with url: /v3/search/works?q=well+logging+interpretation&limit=100&offset=0 (Caused by ResponseError('too many 429 error responses'))
ERROR:__main__:CORE error: HTTPSConnectionPool(host='api.core.ac.uk', port=443): Max retries exceeded with url: /v3/search/works?q=hydraulic+fracturing&limit=100&offset=0 (Caused by ResponseError('too many 429 error responses'))
ERROR:__main__:CORE error: HTTPSConnectionPool(host='api.core.ac.uk', port=443): Max retries exceeded with url: /v3/search/works?q=flow+assurance+pipeline&limit=100&offset=0 (Caused by ResponseError('too many 429 error responses'))
ERROR:__main__:CORE error: HTTPSConnectionPool(host='api.core.ac.uk', po


📊 Running total: 565 records from 1 sources


### 2.7 — DOAJ (Directory of Open Access Journals)

In [ ]:
def scrape_doaj(queries, max_per_query=150):
    """
    DOAJ indexes open-access journals. Free API, no key needed.
    """
    records, seen = [], set()
    base_url = "https://doaj.org/api/search/articles"

    for query in tqdm(queries, desc="DOAJ"):
        page, qcount = 1, 0
        while qcount < max_per_query:
            url = f"{base_url}/{quote_plus(query)}?page={page}&pageSize=100"
            resp = safe_request(url)
            if not resp: break
            try: data = resp.json()
            except: break
            results = data.get('results', [])
            if not results: break

            for item in results:
                bib = item.get('bibjson', {})
                did = item.get('id', '')
                if did in seen: continue
                seen.add(did)

                title = clean_text(bib.get('title', ''))
                abstract = clean_text(bib.get('abstract', ''))
                if not abstract or len(abstract) < 50: continue

                authors = [a.get('name','') for a in bib.get('author', [])]
                journal = bib.get('journal', {}).get('title', '')
                year = bib.get('year', '')

                records.append({
                    "id": generate_id(did), "source": "doaj", "doaj_id": did,
                    "title": title, "abstract": abstract, "authors": authors,
                    "journal": journal, "year": year,
                    "content_type": "research_paper",
                    "text": f"Title: {title}\nJournal: {journal}\n\nAbstract: {abstract}",
                    "scraped_at": datetime.now().isoformat()
                })
                qcount += 1

            page += 1; polite_delay()
            if len(results) < 100: break

    logger.info(f"DOAJ: {len(records)} papers")
    return records

DOAJ_QUERIES = [
    "petroleum engineering", "reservoir simulation", "enhanced oil recovery",
    "drilling engineering", "well completion", "hydraulic fracturing",
    "production optimization oil gas", "flow assurance", "petrophysics",
    "multiphase flow pipeline", "artificial lift", "well testing",
    "formation damage", "sand control", "offshore production",
    "gas processing", "petroleum refining", "carbon capture storage",
]

doaj_records = scrape_doaj(DOAJ_QUERIES, max_per_query=100)
save_records(doaj_records, "07_doaj.jsonl")
log_source("doaj", len(doaj_records))

DOAJ:   0%|          | 0/18 [00:00<?, ?it/s]


📊 Running total: 43,469 records from 5 sources


### 2.8 — Unpaywall (Open Access Full Texts via DOI)

In [ ]:
def enrich_with_unpaywall(records_with_dois, email):
    """
    For records that have DOIs, check Unpaywall for open-access full text.
    This significantly enriches the corpus with full paper content.
    """
    if not email:
        print("⚠️ Set POLITE_EMAIL to use Unpaywall. Skipping.")
        return []

    enriched = []
    dois = []

    # Collect DOIs from all records
    for rec in records_with_dois:
        doi = rec.get('doi', '')
        if doi and doi.startswith('10.'):
            dois.append(doi)
        elif doi and 'doi.org/' in doi:
            dois.append(doi.split('doi.org/')[-1])

    dois = list(set(dois))[:2000]  # Cap at 2000 lookups
    print(f"🔍 Checking {len(dois)} DOIs on Unpaywall...")

    for doi in tqdm(dois, desc="Unpaywall"):
        url = f"https://api.unpaywall.org/v2/{doi}?email={email}"
        resp = safe_request(url)
        if not resp: continue

        try: data = resp.json()
        except: continue

        if not data.get('is_oa'): continue

        # Find best OA location
        best_oa = data.get('best_oa_location', {})
        pdf_url = best_oa.get('url_for_pdf') or best_oa.get('url', '')

        if not pdf_url: continue

        # Try to fetch the page (not PDF, but HTML version)
        landing_url = best_oa.get('url_for_landing_page', pdf_url)
        if landing_url and not landing_url.endswith('.pdf'):
            page_resp = safe_request(landing_url)
            if page_resp:
                text = extract_text_from_html(page_resp.text, landing_url)
                if text and len(text) > 300:
                    # Truncate very long texts
                    words = text.split()
                    if len(words) > 5000:
                        text = ' '.join(words[:5000])

                    enriched.append({
                        "id": generate_id(doi),
                        "source": "unpaywall_fulltext",
                        "doi": doi,
                        "title": clean_text(data.get('title', '')),
                        "content_type": "research_paper_fulltext",
                        "oa_status": data.get('oa_status', ''),
                        "text": f"Title: {clean_text(data.get('title',''))}\n\n{text}",
                        "word_count": len(text.split()),
                        "scraped_at": datetime.now().isoformat()
                    })

        polite_delay(0.5)  # Unpaywall is generous but be polite

    logger.info(f"Unpaywall: Enriched {len(enriched)} papers with full text")
    return enriched

# Collect all records with DOIs from previous sources
all_doi_records = []
for jsonl_file in RAW_DIR.glob("*.jsonl"):
    with jsonlines.open(jsonl_file) as reader:
        for rec in reader:
            if rec.get('doi'): all_doi_records.append(rec)

unpaywall_records = enrich_with_unpaywall(all_doi_records, POLITE_EMAIL)
if unpaywall_records:
    save_records(unpaywall_records, "08_unpaywall.jsonl")
log_source("unpaywall", len(unpaywall_records))

🔍 Checking 2000 DOIs on Unpaywall...


Unpaywall:   0%|          | 0/2000 [00:00<?, ?it/s]

ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:__main__:Request failed (attempt 1/3): HTTPConnectionPool(host='jpme.journals.ekb.eg', port=80): Max retries exceeded with url: /article_39243.html (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x79426a4b5dc0>, 'Connection to jpme.journals.ekb.eg timed out. (connect timeout=30)'))
ERROR:__main__:Request failed (attempt 2/3): HTTPConnectionPool(host='jpme.journals.ekb.eg', port=80): Max retries exceeded with url: /article_39243.html (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x79426a403a10>, 'Connection to jpme.journals.ekb.eg timed out. (connect timeout=30)'))
ERROR:__main__:Request failed (attempt 3/3): HTTPConnectionPool(host='jpme.journals.ekb.eg', port=80): Max retries exceeded with url: /article_39243.html (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x79426a

KeyboardInterrupt: 

In [ ]:
import requests
from urllib.parse import urlparse

# Domains that almost always block scrapers — skip them to save time
BLOCKED_DOMAINS = {
    'doi.org', 'dx.doi.org',  # These just redirect; resolve first
    'onlinelibrary.wiley.com', 'link.springer.com',
    'aip.scitation.org', 'pubs.acs.org', 'ieeexplore.ieee.org',
    'www.sciencedirect.com', 'academic.oup.com',
    'www.nature.com', 'www.tandfonline.com',
}

SCRAPE_HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}


def resolve_doi(doi, timeout=15):
    """Resolve a DOI to its final landing URL without downloading the page."""
    try:
        resp = requests.head(
            f"https://doi.org/{doi}",
            headers=SCRAPE_HEADERS,
            allow_redirects=True,
            timeout=timeout,
        )
        return resp.url
    except Exception:
        return None


def is_blocked_domain(url):
    """Check if the URL's domain is in our known-blocked list."""
    try:
        domain = urlparse(url).netloc.lower()
        return any(domain.endswith(bd) for bd in BLOCKED_DOMAINS)
    except Exception:
        return True


def fetch_page(url, timeout=20):
    """Fetch a page with browser-like headers and short timeout."""
    if is_blocked_domain(url):
        return None
    try:
        resp = requests.get(
            url,
            headers=SCRAPE_HEADERS,
            timeout=timeout,
            allow_redirects=True,
        )
        if resp.status_code == 200 and len(resp.text) > 500:
            return resp
        else:
            logger.warning(f"{resp.status_code} for {url}")
            return None
    except requests.exceptions.RequestException as e:
        logger.warning(f"Fetch failed: {url} — {type(e).__name__}")
        return None


def enrich_with_unpaywall(records_with_dois, email):
    """
    For records that have DOIs, check Unpaywall for open-access full text.
    Skips known-blocked publisher domains to avoid 403 floods.
    """
    if not email:
        print("⚠️ Set POLITE_EMAIL to use Unpaywall. Skipping.")
        return []

    enriched = []
    dois = []

    for rec in records_with_dois:
        doi = rec.get('doi', '')
        if doi and doi.startswith('10.'):
            dois.append(doi)
        elif doi and 'doi.org/' in doi:
            dois.append(doi.split('doi.org/')[-1])

    dois = list(set(dois))[:2000]
    print(f"🔍 Checking {len(dois)} DOIs on Unpaywall...")

    skipped_blocked = 0
    skipped_no_oa = 0
    fetch_failures = 0

    for doi in tqdm(dois, desc="Unpaywall"):
        url = f"https://api.unpaywall.org/v2/{doi}?email={email}"
        resp = safe_request(url)
        if not resp:
            continue

        try:
            data = resp.json()
        except Exception:
            continue

        if not data.get('is_oa'):
            skipped_no_oa += 1
            continue

        best_oa = data.get('best_oa_location') or {}
        landing_url = best_oa.get('url_for_landing_page', '')
        pdf_url = best_oa.get('url_for_pdf', '')

        # Pick the best URL to try — prefer landing page HTML over PDF
        target_url = landing_url or pdf_url
        if not target_url:
            continue

        # If it's a doi.org URL, resolve it first to get the real domain
        if 'doi.org/' in target_url:
            resolved = resolve_doi(doi)
            if resolved:
                target_url = resolved
            else:
                continue

        # Skip if the resolved domain is known to block scrapers
        if is_blocked_domain(target_url):
            skipped_blocked += 1
            continue

        # Skip PDFs (we're only doing HTML extraction here)
        if target_url.lower().endswith('.pdf'):
            # Optionally: add PDF handling with PyMuPDF/pdfplumber later
            continue

        page_resp = fetch_page(target_url)
        if not page_resp:
            fetch_failures += 1
            continue

        text = extract_text_from_html(page_resp.text, target_url)
        if not text or len(text) < 300:
            continue

        words = text.split()
        if len(words) > 5000:
            text = ' '.join(words[:5000])

        enriched.append({
            "id": generate_id(doi),
            "source": "unpaywall_fulltext",
            "doi": doi,
            "title": clean_text(data.get('title', '')),
            "content_type": "research_paper_fulltext",
            "oa_status": data.get('oa_status', ''),
            "text": f"Title: {clean_text(data.get('title',''))}\n\n{text}",
            "word_count": len(words),
            "scraped_at": datetime.now().isoformat(),
        })

        polite_delay(0.5)

    print(f"\n📊 Unpaywall summary:")
    print(f"   ✅ Enriched: {len(enriched)}")
    print(f"   🚫 Blocked domains skipped: {skipped_blocked}")
    print(f"   🔒 Not open access: {skipped_no_oa}")
    print(f"   ❌ Fetch failures: {fetch_failures}")

    logger.info(f"Unpaywall: Enriched {len(enriched)} papers with full text")
    return enriched


# --- Run ---
all_doi_records = []
for jsonl_file in RAW_DIR.glob("*.jsonl"):
    with jsonlines.open(jsonl_file) as reader:
        for rec in reader:
            if rec.get('doi'):
                all_doi_records.append(rec)

unpaywall_records = enrich_with_unpaywall(all_doi_records, POLITE_EMAIL)
if unpaywall_records:
    save_records(unpaywall_records, "08_unpaywall.jsonl")
log_source("unpaywall", len(unpaywall_records))

🔍 Checking 2000 DOIs on Unpaywall...


Unpaywall:   0%|          | 0/2000 [00:00<?, ?it/s]

ERROR:__main__:Request failed (attempt 1/3): 404 Client Error: Not Found for url: https://api.unpaywall.org/v2/10.5555/2048536.2048548?email=you@example.com
ERROR:__main__:Request failed (attempt 2/3): 404 Client Error: Not Found for url: https://api.unpaywall.org/v2/10.5555/2048536.2048548?email=you@example.com
ERROR:__main__:Request failed (attempt 3/3): 404 Client Error: Not Found for url: https://api.unpaywall.org/v2/10.5555/2048536.2048548?email=you@example.com
ERROR:__main__:Request failed (attempt 1/3): 404 Client Error: Not Found for url: https://api.unpaywall.org/v2/10.3303/cet2081181?email=you@example.com
ERROR:__main__:Request failed (attempt 2/3): 404 Client Error: Not Found for url: https://api.unpaywall.org/v2/10.3303/cet2081181?email=you@example.com
ERROR:__main__:Request failed (attempt 3/3): 404 Client Error: Not Found for url: https://api.unpaywall.org/v2/10.3303/cet2081181?email=you@example.com
ERROR:__main__:Request failed (attempt 1/3): 404 Client Error: Not Found 


📊 Unpaywall summary:
   ✅ Enriched: 235
   🚫 Blocked domains skipped: 120
   🔒 Not open access: 1262
   ❌ Fetch failures: 224

📊 Running total: 43,704 records from 6 sources


### 2.9 — PetroWiki (SPE Knowledge Base)

In [ ]:
def scrape_petrowiki():
    records, seen = [], set()
    base_url = "https://petrowiki.spe.org"
    seeds = [
        "/wiki/Reservoir_characterization", "/wiki/Drilling_engineering",
        "/wiki/Well_completions", "/wiki/Production_engineering",
        "/wiki/Reservoir_engineering", "/wiki/Formation_evaluation",
        "/wiki/Enhanced_oil_recovery", "/wiki/Hydraulic_fracturing",
        "/wiki/Artificial_lift", "/wiki/Well_testing",
        "/wiki/Flow_assurance", "/wiki/Offshore_production",
        "/wiki/Petroleum_geology", "/wiki/Petrophysics",
        "/wiki/Geomechanics", "/wiki/Cementing",
        "/wiki/Casing_design", "/wiki/Sand_control",
        "/wiki/Corrosion_in_production", "/wiki/Water_treating",
        "/wiki/Oil_and_gas_separators", "/wiki/PVT_analysis",
        "/wiki/Gas_lift", "/wiki/Electrical_submersible_pumps",
        "/wiki/Sucker-rod_pumping", "/wiki/Horizontal_wells",
        "/wiki/Multilateral_wells", "/wiki/Underbalanced_drilling",
        "/wiki/Coiled_tubing", "/wiki/Wireline_operations",
        "/wiki/Managed_pressure_drilling", "/wiki/Directional_drilling",
        "/wiki/Well_control", "/wiki/Drilling_fluids",
        "/wiki/Rock_mechanics", "/wiki/Seismic_data_acquisition",
        "/wiki/Well_logging", "/wiki/Permeability_determination",
    ]

    # Discover all article URLs
    all_urls = set()
    print("🔍 Discovering PetroWiki articles...")
    for seed in tqdm(seeds, desc="PetroWiki seeds"):
        resp = safe_request(base_url + seed)
        if not resp: continue
        soup = BeautifulSoup(resp.text, 'lxml')
        for link in soup.find_all('a', href=True):
            href = link['href']
            if href.startswith('/wiki/') and ':' not in href and '#' not in href:
                all_urls.add(base_url + href)
        all_urls.add(base_url + seed)
        polite_delay()

    print(f"📋 Found {len(all_urls)} PetroWiki articles")

    for url in tqdm(list(all_urls)[:MAX_ITEMS_PER_SOURCE], desc="Scraping PetroWiki"):
        if url in seen: continue
        seen.add(url)
        resp = safe_request(url)
        if not resp: continue
        soup = BeautifulSoup(resp.text, 'lxml')
        title_tag = soup.find('h1', {'id': 'firstHeading'}) or soup.find('h1')
        title = clean_text(title_tag.get_text()) if title_tag else ''
        if not title: continue
        content = soup.find('div', {'id': 'mw-content-text'}) or soup.find('div', class_='mw-parser-output')
        if not content: continue
        for u in content.find_all(['table','div'], class_=['navbox','metadata','ambox','toc']): u.decompose()
        for u in content.find_all('sup'): u.decompose()
        paragraphs = []
        for p in content.find_all(['p','h2','h3','h4','li']):
            t = clean_text(p.get_text())
            if t and len(t) > 20:
                if p.name in ['h2','h3','h4']:
                    if any(s in t.lower() for s in ['see also','references','external links','bibliography']): break
                    paragraphs.append(f"\n## {t}\n")
                else: paragraphs.append(t)
        full = '\n'.join(paragraphs)
        if len(full) < 100: continue
        records.append({
            "id": generate_id(url), "source": "petrowiki", "title": title,
            "url": url, "content_type": "knowledge_base",
            "text": f"# {title}\n\n{full}", "word_count": len(full.split()),
            "scraped_at": datetime.now().isoformat()
        })
        polite_delay()
    logger.info(f"PetroWiki: {len(records)} articles")
    return records

pw_records = scrape_petrowiki()
save_records(pw_records, "09_petrowiki.jsonl")
log_source("petrowiki", len(pw_records))

🔍 Discovering PetroWiki articles...


PetroWiki seeds:   0%|          | 0/38 [00:00<?, ?it/s]

📋 Found 0 PetroWiki articles


Scraping PetroWiki: 0it [00:00, ?it/s]


📊 Running total: 565 records from 2 sources


In [ ]:
import requests
from urllib.parse import urlparse, quote

SCRAPE_HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Referer": "https://www.google.com/",
    "DNT": "1",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}


def fetch_with_fallback(url, session, timeout=20):
    """
    Try the live URL first. If 403/blocked, try the Wayback Machine cached version.
    """
    # Attempt 1: Live site
    try:
        resp = session.get(url, timeout=timeout)
        if resp.status_code == 200 and len(resp.text) > 500:
            return resp
        elif resp.status_code == 403:
            logger.warning(f"403 on live site, trying Wayback: {url}")
        else:
            logger.warning(f"{resp.status_code}: {url}")
            return None
    except requests.exceptions.RequestException as e:
        logger.warning(f"Live fetch failed: {url} — {type(e).__name__}")

    # Attempt 2: Wayback Machine
    try:
        wb_url = f"https://web.archive.org/web/2024/{url}"
        resp = session.get(wb_url, timeout=timeout)
        if resp.status_code == 200 and len(resp.text) > 500:
            logger.info(f"✅ Got Wayback version: {url}")
            return resp
    except requests.exceptions.RequestException:
        pass

    return None


def scrape_petrowiki():
    records, seen = [], set()
    base_url = "https://petrowiki.spe.org"
    seeds = [
        "/wiki/Reservoir_characterization", "/wiki/Drilling_engineering",
        "/wiki/Well_completions", "/wiki/Production_engineering",
        "/wiki/Reservoir_engineering", "/wiki/Formation_evaluation",
        "/wiki/Enhanced_oil_recovery", "/wiki/Hydraulic_fracturing",
        "/wiki/Artificial_lift", "/wiki/Well_testing",
        "/wiki/Flow_assurance", "/wiki/Offshore_production",
        "/wiki/Petroleum_geology", "/wiki/Petrophysics",
        "/wiki/Geomechanics", "/wiki/Cementing",
        "/wiki/Casing_design", "/wiki/Sand_control",
        "/wiki/Corrosion_in_production", "/wiki/Water_treating",
        "/wiki/Oil_and_gas_separators", "/wiki/PVT_analysis",
        "/wiki/Gas_lift", "/wiki/Electrical_submersible_pumps",
        "/wiki/Sucker-rod_pumping", "/wiki/Horizontal_wells",
        "/wiki/Multilateral_wells", "/wiki/Underbalanced_drilling",
        "/wiki/Coiled_tubing", "/wiki/Wireline_operations",
        "/wiki/Managed_pressure_drilling", "/wiki/Directional_drilling",
        "/wiki/Well_control", "/wiki/Drilling_fluids",
        "/wiki/Rock_mechanics", "/wiki/Seismic_data_acquisition",
        "/wiki/Well_logging", "/wiki/Permeability_determination",
    ]

    # Use a persistent session (keeps cookies, looks more like a real browser)
    session = requests.Session()
    session.headers.update(SCRAPE_HEADERS)

    # Warm up the session — hit the homepage first to collect cookies
    try:
        session.get(base_url, timeout=15)
        polite_delay(1.0)
    except Exception:
        pass

    # Phase 1: Discover article URLs from seed pages
    all_urls = set()
    print("🔍 Discovering PetroWiki articles...")
    for seed in tqdm(seeds, desc="PetroWiki seeds"):
        full_url = base_url + seed
        resp = fetch_with_fallback(full_url, session)
        if not resp:
            continue

        soup = BeautifulSoup(resp.text, 'lxml')
        for link in soup.find_all('a', href=True):
            href = link['href']
            # Normalize Wayback URLs back to petrowiki URLs
            if 'petrowiki.spe.org/wiki/' in href:
                path = '/wiki/' + href.split('/wiki/')[-1]
                href = path
            if href.startswith('/wiki/') and ':' not in href and '#' not in href:
                all_urls.add(base_url + href)
        all_urls.add(full_url)
        polite_delay(1.0)  # Slightly longer delay to be respectful

    print(f"📋 Found {len(all_urls)} PetroWiki articles")

    # Phase 2: Scrape each article
    for url in tqdm(list(all_urls)[:MAX_ITEMS_PER_SOURCE], desc="Scraping PetroWiki"):
        if url in seen:
            continue
        seen.add(url)

        resp = fetch_with_fallback(url, session)
        if not resp:
            continue

        soup = BeautifulSoup(resp.text, 'lxml')

        title_tag = soup.find('h1', {'id': 'firstHeading'}) or soup.find('h1')
        title = clean_text(title_tag.get_text()) if title_tag else ''
        if not title:
            continue

        content = (
            soup.find('div', {'id': 'mw-content-text'})
            or soup.find('div', class_='mw-parser-output')
        )
        if not content:
            continue

        # Remove noise elements
        for u in content.find_all(['table', 'div'], class_=['navbox', 'metadata', 'ambox', 'toc']):
            u.decompose()
        for u in content.find_all('sup'):
            u.decompose()

        paragraphs = []
        for p in content.find_all(['p', 'h2', 'h3', 'h4', 'li']):
            t = clean_text(p.get_text())
            if t and len(t) > 20:
                if p.name in ['h2', 'h3', 'h4']:
                    if any(s in t.lower() for s in ['see also', 'references', 'external links', 'bibliography']):
                        break
                    paragraphs.append(f"\n## {t}\n")
                else:
                    paragraphs.append(t)

        full = '\n'.join(paragraphs)
        if len(full) < 100:
            continue

        records.append({
            "id": generate_id(url),
            "source": "petrowiki",
            "title": title,
            "url": url,
            "content_type": "knowledge_base",
            "text": f"# {title}\n\n{full}",
            "word_count": len(full.split()),
            "scraped_at": datetime.now().isoformat(),
        })
        polite_delay(1.0)

    logger.info(f"PetroWiki: {len(records)} articles")
    return records


pw_records = scrape_petrowiki()
save_records(pw_records, "09_petrowiki.jsonl")
log_source("petrowiki", len(pw_records))

🔍 Discovering PetroWiki articles...


PetroWiki seeds:   0%|          | 0/38 [00:00<?, ?it/s]

📋 Found 0 PetroWiki articles


Scraping PetroWiki: 0it [00:00, ?it/s]


📊 Running total: 1,153 records from 11 sources


### 2.10 — SLB Oilfield Glossary

In [ ]:
def scrape_slb_glossary():
    records, seen = [], set()
    base_url = "https://glossary.slb.com"
    term_urls = []

    print("🔍 Discovering glossary terms...")
    for letter in tqdm('ABCDEFGHIJKLMNOPQRSTUVWXYZ', desc="SLB letters"):
        resp = safe_request(f"{base_url}/terms?letter={letter}")
        if not resp: continue
        soup = BeautifulSoup(resp.text, 'lxml')
        for link in soup.find_all('a', href=True):
            href = link['href']
            if '/terms/' in href:
                full = urljoin(base_url, href)
                if full not in seen:
                    seen.add(full)
                    term_urls.append((full, clean_text(link.get_text())))
        polite_delay()

    print(f"📋 Found {len(term_urls)} terms")
    for url, term in tqdm(term_urls[:MAX_ITEMS_PER_SOURCE], desc="Scraping SLB terms"):
        resp = safe_request(url)
        if not resp: continue
        soup = BeautifulSoup(resp.text, 'lxml')
        defn = soup.find('div', class_='definition') or soup.find('div', class_='term-definition')
        if not defn:
            main = soup.find('main') or soup.find('article')
            if main: definition = clean_text(main.get_text())
            else: continue
        else: definition = clean_text(defn.get_text())
        if len(definition) < 20: continue
        disc_tag = soup.find('span', class_='discipline') or soup.find('div', class_='discipline')
        discipline = clean_text(disc_tag.get_text()) if disc_tag else ''
        records.append({
            "id": generate_id(url), "source": "slb_glossary",
            "term": term, "definition": definition, "discipline": discipline,
            "url": url, "content_type": "technical_definition",
            "text": f"Term: {term}\nDiscipline: {discipline}\n\nDefinition: {definition}",
            "scraped_at": datetime.now().isoformat()
        })
        polite_delay()
    logger.info(f"SLB Glossary: {len(records)} terms")
    return records

slb_records = scrape_slb_glossary()
save_records(slb_records, "10_slb_glossary.jsonl")
log_source("slb_glossary", len(slb_records))

🔍 Discovering glossary terms...


SLB letters:   0%|          | 0/26 [00:00<?, ?it/s]

ERROR:__main__:Request failed (attempt 1/3): 404 Client Error: 404 File Not Found for url: https://glossary.slb.com/terms?letter=A
ERROR:__main__:Request failed (attempt 2/3): 404 Client Error: 404 File Not Found for url: https://glossary.slb.com/terms?letter=A
ERROR:__main__:Request failed (attempt 3/3): 404 Client Error: 404 File Not Found for url: https://glossary.slb.com/terms?letter=A
ERROR:__main__:Request failed (attempt 1/3): 404 Client Error: 404 File Not Found for url: https://glossary.slb.com/terms?letter=B
ERROR:__main__:Request failed (attempt 2/3): 404 Client Error: 404 File Not Found for url: https://glossary.slb.com/terms?letter=B
ERROR:__main__:Request failed (attempt 3/3): 404 Client Error: 404 File Not Found for url: https://glossary.slb.com/terms?letter=B
ERROR:__main__:Request failed (attempt 1/3): 404 Client Error: 404 File Not Found for url: https://glossary.slb.com/terms?letter=C
ERROR:__main__:Request failed (attempt 2/3): 404 Client Error: 404 File Not Found f

📋 Found 0 terms


Scraping SLB terms: 0it [00:00, ?it/s]


📊 Running total: 565 records from 3 sources


In [ ]:
import requests
from urllib.parse import urljoin

SCRAPE_HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/",
}


def scrape_slb_glossary():
    records, seen = [], set()
    base_url = "https://glossary.slb.com"
    term_urls = set()

    session = requests.Session()
    session.headers.update(SCRAPE_HEADERS)

    # Warm up session — grab homepage cookies
    try:
        session.get(f"{base_url}/en/", timeout=15)
        polite_delay(1.0)
    except Exception:
        pass

    print("🔍 Discovering glossary terms...")

    # Phase 1: Try letter index pages with the NEW URL pattern
    for letter in tqdm('abcdefghijklmnopqrstuvwxyz', desc="SLB letters"):
        url = f"{base_url}/en/terms/{letter}"
        try:
            resp = session.get(url, timeout=20)
            if resp.status_code != 200:
                logger.warning(f"{resp.status_code}: {url}")
                continue
        except requests.exceptions.RequestException as e:
            logger.warning(f"Failed: {url} — {type(e).__name__}")
            continue

        soup = BeautifulSoup(resp.text, 'lxml')
        for link in soup.find_all('a', href=True):
            href = link['href']
            # Match term URLs like /en/terms/a/abandoned-well or /terms/a/abandoned-well
            if '/terms/' in href and href.count('/') >= 3:
                # Skip the letter index links themselves (e.g., /en/terms/a)
                parts = href.rstrip('/').split('/')
                if len(parts) >= 2 and len(parts[-1]) > 1:  # actual term, not just a letter
                    full_url = urljoin(base_url, href)
                    # Normalize to /en/ prefix
                    if '/en/' not in full_url:
                        full_url = full_url.replace('/terms/', '/en/terms/')
                    term_urls.add(full_url)
        polite_delay(1.0)

    # Phase 2: If letter pages didn't yield results (JS-rendered),
    # crawl from known seed terms using cross-links
    if len(term_urls) < 50:
        print("⚠️ Letter pages may be JS-rendered. Crawling via cross-links...")
        seed_terms = [
            "petroleum", "drilling", "completion", "permeability", "porosity",
            "reservoir", "fracture", "casing", "cement", "wellbore",
            "production", "injection", "viscosity", "saturation", "mudcake",
            "formation", "logging", "seismic", "geophone", "hydrocarbon",
            "acidizing", "gravel_pack", "packer", "tubing", "annulus",
            "blowout", "kelly", "drillstring", "bit", "dogleg",
            "deviation", "inclination", "azimuth", "torque", "standpipe",
            "mud_weight", "lost_circulation", "kick", "bop", "choke",
        ]
        for term in seed_terms:
            letter = term[0]
            term_urls.add(f"{base_url}/en/terms/{letter}/{term}")

        # Crawl cross-links from each seed to discover more terms
        discovered = set()
        for url in tqdm(list(term_urls), desc="Crawling seed terms"):
            try:
                resp = session.get(url, timeout=20)
                if resp.status_code != 200:
                    continue
            except Exception:
                continue

            soup = BeautifulSoup(resp.text, 'lxml')
            for link in soup.find_all('a', href=True):
                href = link['href']
                if '/en/terms/' in href or '/terms/' in href:
                    parts = href.rstrip('/').split('/')
                    if len(parts) >= 2 and len(parts[-1]) > 1:
                        full_url = urljoin(base_url, href)
                        if '/en/' not in full_url:
                            full_url = full_url.replace('/terms/', '/en/terms/')
                        discovered.add(full_url)
            polite_delay(0.5)

        term_urls.update(discovered)

    print(f"📋 Found {len(term_urls)} terms")

    # Phase 3: Scrape each term page
    for url in tqdm(list(term_urls)[:MAX_ITEMS_PER_SOURCE], desc="Scraping SLB terms"):
        if url in seen:
            continue
        seen.add(url)

        try:
            resp = session.get(url, timeout=20)
            if resp.status_code != 200:
                continue
        except requests.exceptions.RequestException:
            continue

        soup = BeautifulSoup(resp.text, 'lxml')

        # Extract term name from <h1> — new site uses bold term in h1
        title_tag = soup.find('h1')
        if not title_tag:
            continue
        # The page has two h1s: "Explore the Energy Glossary" and the term name
        h1_tags = soup.find_all('h1')
        term = ''
        for h1 in h1_tags:
            text = clean_text(h1.get_text())
            if text and text.lower() != 'explore the energy glossary':
                term = text
                break
        if not term:
            continue

        # Extract definitions — they follow <strong> tags with pattern "1. n. [Discipline]"
        # Get the main content area
        main_content = soup.find('main') or soup.find('article') or soup.find('body')
        if not main_content:
            continue

        # Collect all text after the term heading
        definitions = []
        discipline = ''

        # Look for definition blocks with discipline tags
        for strong in main_content.find_all('strong'):
            strong_text = clean_text(strong.get_text())
            # Match patterns like "1. vt. [Geology]" or "2. n. [Drilling]"
            if strong_text and '[' in strong_text and ']' in strong_text:
                # Extract discipline
                disc = strong_text.split('[')[-1].split(']')[0]
                if not discipline:
                    discipline = disc
                definitions.append(f"[{disc}]")

        # Get all paragraph text from the content area
        paragraphs = []
        for p in main_content.find_all(['p', 'li']):
            t = clean_text(p.get_text())
            if t and len(t) > 15:
                # Skip navigation/boilerplate text
                if any(skip in t.lower() for skip in [
                    'look up terms', 'log in', 'sign up', 'premium content',
                    'account permissions', 'share this', 'privacy',
                    'terms of service', 'schlumberger limited',
                    'explore the energy glossary', 'sign in to access',
                ]):
                    continue
                paragraphs.append(t)

        definition = '\n'.join(paragraphs)
        if len(definition) < 20:
            continue

        records.append({
            "id": generate_id(url),
            "source": "slb_glossary",
            "term": term,
            "definition": definition,
            "discipline": discipline,
            "url": url,
            "content_type": "technical_definition",
            "text": f"Term: {term}\nDiscipline: {discipline}\n\nDefinition: {definition}",
            "scraped_at": datetime.now().isoformat(),
        })
        polite_delay(0.5)

    logger.info(f"SLB Glossary: {len(records)} terms")
    return records


slb_records = scrape_slb_glossary()
save_records(slb_records, "10_slb_glossary.jsonl")
log_source("slb_glossary", len(slb_records))

🔍 Discovering glossary terms...


SLB letters:   0%|          | 0/26 [00:00<?, ?it/s]

⚠️ Letter pages may be JS-rendered. Crawling via cross-links...


Crawling seed terms:   0%|          | 0/40 [00:00<?, ?it/s]

📋 Found 495 terms


Scraping SLB terms:   0%|          | 0/495 [00:00<?, ?it/s]


📊 Running total: 1,608 records from 11 sources


### 2.11 — Wikipedia

In [ ]:
def scrape_wikipedia(topics):
    records, seen = [], set()
    base_url = "https://en.wikipedia.org/w/api.php"
    for topic in tqdm(topics, desc="Wikipedia"):
        resp = safe_request(base_url, params={"action":"query","list":"search","srsearch":topic,"srlimit":15,"format":"json"})
        if not resp: continue
        try: data = resp.json()
        except: continue
        for result in data.get('query',{}).get('search',[]):
            title = result.get('title','')
            if title in seen: continue
            seen.add(title)
            cr = safe_request(base_url, params={"action":"query","titles":title,"prop":"extracts","exintro":False,"explaintext":True,"format":"json"})
            if not cr: continue
            try: cd = cr.json()
            except: continue
            for pid, page in cd.get('query',{}).get('pages',{}).items():
                if pid == '-1': continue
                extract = page.get('extract','')
                if not extract or len(extract) < 200: continue
                for sh in ['== See also ==','== References ==','== External links ==','== Notes ==']:
                    idx = extract.find(sh)
                    if idx != -1: extract = extract[:idx]
                extract = clean_text(extract)
                records.append({
                    "id": generate_id(title), "source": "wikipedia", "title": title,
                    "url": f"https://en.wikipedia.org/wiki/{title.replace(' ','_')}",
                    "content_type": "encyclopedia",
                    "text": f"# {title}\n\n{extract}", "word_count": len(extract.split()),
                    "scraped_at": datetime.now().isoformat()
                })
            polite_delay()
    logger.info(f"Wikipedia: {len(records)} articles")
    return records

wiki_records = scrape_wikipedia(WIKI_TOPICS)
save_records(wiki_records, "11_wikipedia.jsonl")
log_source("wikipedia", len(wiki_records))

Wikipedia:   0%|          | 0/119 [00:00<?, ?it/s]


📊 Running total: 565 records from 4 sources


In [ ]:
def scrape_wikipedia(topics):
    records, seen = [], set()
    base_url = "https://en.wikipedia.org/w/api.php"

    session = requests.Session()
    # Wikipedia requires a descriptive User-Agent per their API etiquette:
    # https://meta.wikimedia.org/wiki/User-Agent_policy
    session.headers.update({
        "User-Agent": "PetroleumDataCollector/1.0 (research project; contact@example.com)",
        "Accept": "application/json",
    })

    for topic in tqdm(topics, desc="Wikipedia"):
        try:
            resp = session.get(base_url, params={
                "action": "query",
                "list": "search",
                "srsearch": topic,
                "srlimit": 15,
                "format": "json",
            }, timeout=20)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            logger.warning(f"Search failed for '{topic}': {e}")
            continue

        for result in data.get('query', {}).get('search', []):
            title = result.get('title', '')
            if title in seen:
                continue
            seen.add(title)

            try:
                cr = session.get(base_url, params={
                    "action": "query",
                    "titles": title,
                    "prop": "extracts",
                    "exintro": False,
                    "explaintext": True,
                    "format": "json",
                }, timeout=20)
                cr.raise_for_status()
                cd = cr.json()
            except Exception as e:
                logger.warning(f"Extract failed for '{title}': {e}")
                continue

            for pid, page in cd.get('query', {}).get('pages', {}).items():
                if pid == '-1':
                    continue
                extract = page.get('extract', '')
                if not extract or len(extract) < 200:
                    continue

                # Trim trailing reference sections
                for sh in ['== See also ==', '== References ==', '== External links ==', '== Notes ==']:
                    idx = extract.find(sh)
                    if idx != -1:
                        extract = extract[:idx]

                extract = clean_text(extract)

                records.append({
                    "id": generate_id(title),
                    "source": "wikipedia",
                    "title": title,
                    "url": f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}",
                    "content_type": "encyclopedia",
                    "text": f"# {title}\n\n{extract}",
                    "word_count": len(extract.split()),
                    "scraped_at": datetime.now().isoformat(),
                })

            polite_delay(0.2)  # Wikipedia API allows ~200 req/s for good bots, but be polite

    logger.info(f"Wikipedia: {len(records)} articles")
    return records


wiki_records = scrape_wikipedia(WIKI_TOPICS)
save_records(wiki_records, "11_wikipedia.jsonl")
log_source("wikipedia", len(wiki_records))

Wikipedia:   0%|          | 0/119 [00:00<?, ?it/s]


📊 Running total: 2,697 records from 11 sources


### 2.12 — EIA Data & Articles

In [ ]:
def scrape_eia():
    records = []
    seed_urls = [
        "https://www.eia.gov/petroleum/", "https://www.eia.gov/petroleum/drilling/",
        "https://www.eia.gov/petroleum/production/", "https://www.eia.gov/naturalgas/",
        "https://www.eia.gov/analysis/", "https://www.eia.gov/todayinenergy/",
        "https://www.eia.gov/petroleum/supply/weekly/",
    ]
    discovered = set(seed_urls)

    print("🔍 Discovering EIA pages...")
    for url in tqdm(seed_urls, desc="EIA seeds"):
        resp = safe_request(url)
        if not resp: continue
        soup = BeautifulSoup(resp.text, 'lxml')
        for link in soup.find_all('a', href=True):
            full = urljoin(url, link['href'])
            if full.startswith('https://www.eia.gov') and any(k in full.lower() for k in ['petroleum','oil','gas','crude','drilling','production','energy']):
                discovered.add(full)
        polite_delay()

    print(f"📋 Found {len(discovered)} EIA URLs")
    for url in tqdm(list(discovered)[:600], desc="Scraping EIA"):
        resp = safe_request(url)
        if not resp: continue
        text = extract_text_from_html(resp.text, url)
        if text and len(text) > 200:
            soup = BeautifulSoup(resp.text, 'lxml')
            title = clean_text(soup.find('title').get_text()) if soup.find('title') else ''
            records.append({
                "id": generate_id(url), "source": "eia", "title": title,
                "url": url, "content_type": "government_report",
                "text": f"# {title}\n\n{text}", "scraped_at": datetime.now().isoformat()
            })
        polite_delay()
    logger.info(f"EIA: {len(records)} articles")
    return records

eia_records = scrape_eia()
save_records(eia_records, "12_eia.jsonl")
log_source("eia", len(eia_records))

🔍 Discovering EIA pages...


EIA seeds:   0%|          | 0/7 [00:00<?, ?it/s]

📋 Found 443 EIA URLs


Scraping EIA:   0%|          | 0/443 [00:00<?, ?it/s]

ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tre


📊 Running total: 1,007 records from 5 sources


### 2.13 — EPA, USGS, Industry Sites

In [ ]:
def scrape_web_sources(source_configs):
    """
    Generic web scraper for multiple sources: EPA, USGS, Rigzone, JPT, World Oil, etc.
    """
    all_records = []

    for config in source_configs:
        name = config['name']
        seed_urls = config['seeds']
        keywords = config.get('keywords', ['oil','gas','petroleum','drilling','reservoir'])
        max_pages = config.get('max_pages', 300)
        source_tag = config['tag']
        content_type = config.get('content_type', 'industry_article')

        records = []
        discovered = set(seed_urls)

        print(f"\n🔍 Discovering {name} pages...")
        for url in tqdm(seed_urls, desc=f"{name} seeds"):
            resp = safe_request(url)
            if not resp: continue
            soup = BeautifulSoup(resp.text, 'lxml')
            for link in soup.find_all('a', href=True):
                full = urljoin(url, link['href'])
                base_domain = url.split('/')[2]
                if base_domain in full and any(k in full.lower() for k in keywords):
                    discovered.add(full)
            polite_delay()

        print(f"📋 {name}: Found {len(discovered)} URLs")
        for url in tqdm(list(discovered)[:max_pages], desc=f"Scraping {name}"):
            resp = safe_request(url)
            if not resp: continue
            text = extract_text_from_html(resp.text, url)
            if text and len(text) > 200:
                soup = BeautifulSoup(resp.text, 'lxml')
                title = clean_text(soup.find('title').get_text()) if soup.find('title') else ''
                records.append({
                    "id": generate_id(url), "source": source_tag, "title": title,
                    "url": url, "content_type": content_type,
                    "text": f"# {title}\n\n{text}", "scraped_at": datetime.now().isoformat()
                })
            polite_delay()

        if records:
            save_records(records, f"13_{source_tag}.jsonl")
        log_source(source_tag, len(records))
        all_records.extend(records)

    return all_records

WEB_SOURCES = [
    {
        "name": "EPA Oil & Gas",
        "tag": "epa",
        "seeds": ["https://www.epa.gov/oil-and-gas-extraction", "https://www.epa.gov/uog",
                  "https://www.epa.gov/natural-gas-star-program", "https://www.epa.gov/offshore-oil-and-gas"],
        "keywords": ["oil", "gas", "petroleum", "drilling", "fracking", "extraction", "offshore"],
        "content_type": "government_report", "max_pages": 300,
    },
    {
        "name": "USGS Energy",
        "tag": "usgs",
        "seeds": ["https://www.usgs.gov/programs/energy-resources-program",
                  "https://www.usgs.gov/centers/central-energy-resources-science-center",
                  "https://www.usgs.gov/mission-areas/energy-and-minerals"],
        "keywords": ["oil", "gas", "petroleum", "energy", "geological", "assessment", "basin"],
        "content_type": "geological_report", "max_pages": 300,
    },
    {
        "name": "Rigzone",
        "tag": "rigzone",
        "seeds": ["https://www.rigzone.com/news/", "https://www.rigzone.com/training/"],
        "keywords": ["oil", "gas", "drilling", "offshore", "production", "subsea", "completion"],
        "content_type": "industry_article", "max_pages": 400,
    },
    {
        "name": "JPT (SPE)",
        "tag": "jpt",
        "seeds": ["https://jpt.spe.org/", "https://jpt.spe.org/drilling",
                  "https://jpt.spe.org/production-operations", "https://jpt.spe.org/reservoir"],
        "keywords": ["oil", "gas", "petroleum", "drilling", "reservoir", "completion", "production"],
        "content_type": "industry_article", "max_pages": 400,
    },
]

web_records = scrape_web_sources(WEB_SOURCES)


🔍 Discovering EPA Oil & Gas pages...


EPA Oil & Gas seeds:   0%|          | 0/4 [00:00<?, ?it/s]

ERROR:__main__:Request failed (attempt 1/3): 404 Client Error: Not Found for url: https://www.epa.gov/oil-and-gas-extraction
ERROR:__main__:Request failed (attempt 2/3): 404 Client Error: Not Found for url: https://www.epa.gov/oil-and-gas-extraction
ERROR:__main__:Request failed (attempt 3/3): 404 Client Error: Not Found for url: https://www.epa.gov/oil-and-gas-extraction
ERROR:__main__:Request failed (attempt 1/3): 404 Client Error: Not Found for url: https://www.epa.gov/uog
ERROR:__main__:Request failed (attempt 2/3): 404 Client Error: Not Found for url: https://www.epa.gov/uog
ERROR:__main__:Request failed (attempt 3/3): 404 Client Error: Not Found for url: https://www.epa.gov/uog
ERROR:__main__:Request failed (attempt 1/3): 404 Client Error: Not Found for url: https://www.epa.gov/offshore-oil-and-gas
ERROR:__main__:Request failed (attempt 2/3): 404 Client Error: Not Found for url: https://www.epa.gov/offshore-oil-and-gas
ERROR:__main__:Request failed (attempt 3/3): 404 Client Error

📋 EPA Oil & Gas: Found 19 URLs


Scraping EPA Oil & Gas:   0%|          | 0/19 [00:00<?, ?it/s]

...

ERROR:__main__:Request failed (attempt 1/3): 404 Client Error: Not Found for url: https://jpt.spe.org/drilling
ERROR:__main__:Request failed (attempt 2/3): 404 Client Error: Not Found for url: https://jpt.spe.org/drilling
ERROR:__main__:Request failed (attempt 3/3): 404 Client Error: Not Found for url: https://jpt.spe.org/drilling
ERROR:__main__:Request failed (attempt 1/3): 404 Client Error: Not Found for url: https://jpt.spe.org/reservoir
ERROR:__main__:Request failed (attempt 2/3): 404 Client Error: Not Found for url: https://jpt.spe.org/reservoir
ERROR:__main__:Request failed (attempt 3/3): 404 Client Error: Not Found for url: https://jpt.spe.org/reservoir


📋 JPT (SPE): Found 51 URLs


Scraping JPT (SPE):   0%|          | 0/51 [00:00<?, ?it/s]

ERROR:__main__:Request failed (attempt 1/3): 404 Client Error: Not Found for url: https://jpt.spe.org/drilling
ERROR:__main__:Request failed (attempt 2/3): 404 Client Error: Not Found for url: https://jpt.spe.org/drilling
ERROR:__main__:Request failed (attempt 3/3): 404 Client Error: Not Found for url: https://jpt.spe.org/drilling
ERROR:__main__:Request failed (attempt 1/3): 404 Client Error: Not Found for url: https://jpt.spe.org/reservoir
ERROR:__main__:Request failed (attempt 2/3): 404 Client Error: Not Found for url: https://jpt.spe.org/reservoir
ERROR:__main__:Request failed (attempt 3/3): 404 Client Error: Not Found for url: https://jpt.spe.org/reservoir



📊 Running total: 1,145 records from 9 sources


### 2.14 — OnePetro / SPE Papers

In [ ]:
def scrape_onepetro(queries, max_per_query=100):
    records, seen = [], set()
    base_url = "https://onepetro.org"

    for query in tqdm(queries, desc="OnePetro"):
        page, qcount = 1, 0
        while qcount < max_per_query:
            resp = safe_request(f"{base_url}/search", params={"q": query, "page": page})
            if not resp: break
            soup = BeautifulSoup(resp.text, 'lxml')
            results = soup.find_all('div', class_='search-result') or soup.find_all('div', class_='item-info') or soup.find_all('li', class_='search-result-item')
            if not results: break
            for r in results:
                link = r.find('a', href=True)
                if not link: continue
                purl = urljoin(base_url, link['href'])
                if purl in seen: continue
                seen.add(purl)
                title = clean_text(link.get_text())
                abs_tag = r.find('div', class_='abstract') or r.find('p', class_='snippet') or r.find('div', class_='description')
                abstract = clean_text(abs_tag.get_text()) if abs_tag else ''
                if not abstract or len(abstract) < 50:
                    pr = safe_request(purl)
                    if pr:
                        ps = BeautifulSoup(pr.text, 'lxml')
                        ad = ps.find('section', class_='abstract') or ps.find('div', class_='abstract')
                        if ad: abstract = clean_text(ad.get_text())
                    polite_delay()
                if not abstract or len(abstract) < 50: continue
                records.append({
                    "id": generate_id(purl), "source": "onepetro", "title": title,
                    "abstract": abstract, "url": purl, "content_type": "research_paper",
                    "text": f"Title: {title}\n\nAbstract: {abstract}",
                    "scraped_at": datetime.now().isoformat()
                })
                qcount += 1
            page += 1; polite_delay()
            if len(results) < 10: break
    logger.info(f"OnePetro: {len(records)} papers")
    return records

ONEPETRO_QUERIES = [
    "reservoir simulation history matching", "production optimization artificial lift",
    "drilling rate of penetration", "well completion horizontal", "hydraulic fracture design",
    "flow assurance deepwater", "machine learning petroleum", "digital twin oil gas",
    "FPSO production", "subsea tieback", "well integrity management",
    "formation damage", "water management production", "real-time drilling",
    "multiphase flow metering", "reservoir characterization geostatistics",
    "polymer flooding EOR", "gas condensate deliverability", "sand management",
    "carbon capture storage", "ESP failure analysis", "gas lift optimization",
    "coiled tubing intervention", "managed pressure drilling",
    "wellbore stability geomechanics", "cementing best practices",
    "casing design deepwater", "drill string dynamics",
    "LWD MWD interpretation", "wireline logging evaluation",
]

op_records = scrape_onepetro(ONEPETRO_QUERIES, max_per_query=80)
save_records(op_records, "14_onepetro.jsonl")
log_source("onepetro", len(op_records))

OnePetro:   0%|          | 0/30 [00:00<?, ?it/s]


📊 Running total: 1,145 records from 10 sources


In [ ]:
def scrape_onepetro_via_apis(queries, max_per_query=80):
    """
    Get SPE/petroleum paper metadata via free open APIs instead of
    scraping OnePetro directly (which blocks bots).

    Uses:
    1. OpenAlex API (primary) — free, no key needed, indexes 250M+ works
    2. Semantic Scholar API (fallback) — free, good abstract coverage
    """
    records, seen_dois = [], set()

    session = requests.Session()
    session.headers.update({
        "User-Agent": "PetroleumDataCollector/1.0 (research project; mailto:contact@example.com)",
        "Accept": "application/json",
    })

    # ── Primary: OpenAlex ──────────────────────────────────────────────
    print("📚 Searching OpenAlex for petroleum engineering papers...")
    for query in tqdm(queries, desc="OpenAlex"):
        cursor = "*"
        qcount = 0

        while qcount < max_per_query and cursor:
            params = {
                "search": query,
                "filter": "type:article|proceedings-article",
                "select": "id,doi,title,abstract_inverted_index,authorships,"
                          "publication_year,primary_location,cited_by_count",
                "sort": "relevance_score:desc",
                "per_page": 50,
                "cursor": cursor,
                "mailto": POLITE_EMAIL or "contact@example.com",
            }

            try:
                resp = session.get("https://api.openalex.org/works", params=params, timeout=30)
                resp.raise_for_status()
                data = resp.json()
            except Exception as e:
                logger.warning(f"OpenAlex search failed for '{query}': {e}")
                break

            results = data.get("results", [])
            if not results:
                break

            for work in results:
                doi = work.get("doi", "")
                title = work.get("title", "")
                if not title:
                    continue

                # Deduplicate by DOI
                if doi and doi in seen_dois:
                    continue
                if doi:
                    seen_dois.add(doi)

                # Reconstruct abstract from inverted index
                abstract = ""
                inv_idx = work.get("abstract_inverted_index")
                if inv_idx:
                    abstract = reconstruct_abstract(inv_idx)

                if not abstract or len(abstract) < 50:
                    continue

                # Extract metadata
                year = work.get("publication_year", "")
                cited = work.get("cited_by_count", 0)

                # Get source/journal name
                source_name = ""
                primary_loc = work.get("primary_location") or {}
                source_obj = primary_loc.get("source") or {}
                source_name = source_obj.get("display_name", "")

                # Get authors
                authors = []
                for auth in (work.get("authorships") or [])[:5]:
                    name = auth.get("author", {}).get("display_name", "")
                    if name:
                        authors.append(name)

                # Build OnePetro URL from DOI if it's an SPE paper
                url = doi or work.get("id", "")

                records.append({
                    "id": generate_id(doi or title),
                    "source": "onepetro_openalex",
                    "title": clean_text(title),
                    "abstract": clean_text(abstract),
                    "authors": "; ".join(authors),
                    "year": year,
                    "cited_by_count": cited,
                    "journal": source_name,
                    "doi": doi,
                    "url": url,
                    "content_type": "research_paper",
                    "text": f"Title: {clean_text(title)}\nAuthors: {'; '.join(authors)}\n"
                            f"Year: {year}\nSource: {source_name}\n\n"
                            f"Abstract: {clean_text(abstract)}",
                    "scraped_at": datetime.now().isoformat(),
                })
                qcount += 1

            # Pagination
            meta = data.get("meta", {})
            cursor = meta.get("next_cursor")
            if not cursor or qcount >= max_per_query:
                break

            polite_delay(0.2)  # OpenAlex is generous, ~10 req/s with mailto

    # ── Fallback: Semantic Scholar (for queries with few results) ──────
    if len(records) < len(queries) * 10:
        print("📖 Supplementing with Semantic Scholar...")
        for query in tqdm(queries, desc="Semantic Scholar"):
            try:
                resp = session.get(
                    "https://api.semanticscholar.org/graph/v1/paper/search",
                    params={
                        "query": query,
                        "limit": 30,
                        "fields": "title,abstract,authors,year,citationCount,"
                                  "externalIds,journal,url",
                    },
                    timeout=20,
                )
                resp.raise_for_status()
                data = resp.json()
            except Exception as e:
                logger.warning(f"Semantic Scholar failed for '{query}': {e}")
                polite_delay(1.0)
                continue

            for paper in data.get("data", []):
                doi = (paper.get("externalIds") or {}).get("DOI", "")
                if doi and doi in seen_dois:
                    continue
                if doi:
                    seen_dois.add(doi)

                title = paper.get("title", "")
                abstract = paper.get("abstract", "")
                if not title or not abstract or len(abstract) < 50:
                    continue

                authors = [a.get("name", "") for a in (paper.get("authors") or [])[:5]]
                journal = (paper.get("journal") or {}).get("name", "")
                year = paper.get("year", "")

                records.append({
                    "id": generate_id(doi or title),
                    "source": "onepetro_semanticscholar",
                    "title": clean_text(title),
                    "abstract": clean_text(abstract),
                    "authors": "; ".join(authors),
                    "year": year,
                    "cited_by_count": paper.get("citationCount", 0),
                    "journal": journal,
                    "doi": doi,
                    "url": paper.get("url", ""),
                    "content_type": "research_paper",
                    "text": f"Title: {clean_text(title)}\nAuthors: {'; '.join(authors)}\n"
                            f"Year: {year}\nSource: {journal}\n\n"
                            f"Abstract: {clean_text(abstract)}",
                    "scraped_at": datetime.now().isoformat(),
                })

            polite_delay(1.0)  # Semantic Scholar: ~1 req/s without key

    logger.info(f"OnePetro (via APIs): {len(records)} papers")
    return records


def reconstruct_abstract(inverted_index):
    """
    OpenAlex stores abstracts as inverted indexes: {"word": [positions]}.
    Reconstruct the original text.
    """
    if not inverted_index:
        return ""
    word_positions = []
    for word, positions in inverted_index.items():
        for pos in positions:
            word_positions.append((pos, word))
    word_positions.sort(key=lambda x: x[0])
    return " ".join(w for _, w in word_positions)


# Use the same queries
ONEPETRO_QUERIES = [
    "reservoir simulation history matching", "production optimization artificial lift",
    "drilling rate of penetration", "well completion horizontal", "hydraulic fracture design",
    "flow assurance deepwater", "machine learning petroleum", "digital twin oil gas",
    "FPSO production", "subsea tieback", "well integrity management",
    "formation damage", "water management production", "real-time drilling",
    "multiphase flow metering", "reservoir characterization geostatistics",
    "polymer flooding EOR", "gas condensate deliverability", "sand management",
    "carbon capture storage", "ESP failure analysis", "gas lift optimization",
    "coiled tubing intervention", "managed pressure drilling",
    "wellbore stability geomechanics", "cementing best practices",
    "casing design deepwater", "drill string dynamics",
    "LWD MWD interpretation", "wireline logging evaluation",
]

op_records = scrape_onepetro_via_apis(ONEPETRO_QUERIES, max_per_query=80)
save_records(op_records, "14_onepetro.jsonl")
log_source("onepetro", len(op_records))

📚 Searching OpenAlex for petroleum engineering papers...


OpenAlex:   0%|          | 0/30 [00:00<?, ?it/s]


📊 Running total: 5,668 records from 11 sources


### 2.15 — Dataset Documentation (Volve, FORCE, Kansas GS)

In [ ]:
def collect_dataset_docs():
    records = []

    # Manually curated high-quality dataset descriptions
    datasets = [
        {
            "title": "Volve Field Dataset - Equinor",
            "source": "volve_dataset",
            "text": """The Volve field dataset was released by Equinor in 2018 as the world's first complete open-source oil field dataset. The Volve field is located in the North Sea, approximately 200 km west of Stavanger, Norway. It was discovered in 1993 and produced from 2008 to 2016. The dataset contains: Well logs (LAS files) for exploration and production wells including gamma ray, resistivity, neutron porosity, bulk density, sonic, and caliper curves. Production data includes daily and monthly rates for oil, gas, water, and injection volumes for all wells. Well reports and completion summaries document the design and installation of well equipment. Seismic data includes 3D volumes for structural interpretation. Reservoir simulation models built in Eclipse format are included with history-matched parameters. Geological reports provide formation descriptions, depositional environments, and structural maps. PVT data includes fluid composition, formation volume factors, viscosity measurements, and phase diagrams. Well test data includes build-up and drawdown test results with pressure transient analysis. Key wells include 15/9-F-1, 15/9-F-4, 15/9-F-5, 15/9-F-11, 15/9-F-12, 15/9-F-14, 15/9-F-15. The reservoir is in the Hugin Formation (Upper Jurassic) at approximately 2,750-2,900m depth, consisting of shallow marine sandstones. Peak production was approximately 56,000 boe/d. Total recovery was about 63 million barrels of oil equivalent. This dataset is widely used for machine learning research including production forecasting, well log prediction, reservoir characterization, history matching, and anomaly detection studies."""
        },
        {
            "title": "FORCE 2020 Machine Learning Competition Dataset",
            "source": "force_2020",
            "text": """The FORCE 2020 Machine Learning Competition focused on lithology prediction from well logs on the Norwegian Continental Shelf. Organized by FORCE (Forum for Reservoir Characterization, Reservoir Engineering and Exploration), the dataset includes well log data from 118 training wells and 10 blind test wells. Available log curves include GR (Gamma Ray), RHOB (Bulk Density), NPHI (Neutron Porosity), DTC (Compressional Sonic), DTS (Shear Sonic), PEF (Photoelectric Factor), CALI (Caliper), and derived logs. Lithology classes include Sandstone, Shale/Mudstone, Limestone, Chalk, Marl, Dolomite, Anhydrite, Halite, Coal, Basement, and Tuff. Additional features include well coordinates, formation names, group names, and measured depth. The dataset challenges participants to build models that can accurately classify lithology from standard wireline log measurements. Common approaches include Random Forest, Gradient Boosting (XGBoost, LightGBM), neural networks (1D CNN, LSTM), and ensemble methods. Key challenges include class imbalance, missing log data, depth alignment, and geological variability across different wells and formations. The competition scoring used a penalty matrix that weighted certain misclassifications more heavily than others, reflecting the geological cost of confusing certain lithologies."""
        },
        {
            "title": "Kansas Geological Survey Oil and Gas Database",
            "source": "kansas_gs",
            "text": """The Kansas Geological Survey (KGS) maintains extensive publicly available databases of oil and gas production, well logs, and geological data for the state of Kansas. The oil and gas production database contains monthly production records dating back decades for thousands of wells across multiple producing formations. Key formations include the Arbuckle Group (Cambrian-Ordovician carbonates), Lansing-Kansas City Group (Pennsylvanian limestones), Mississippian Chat, and various Pennsylvanian sandstones. The database includes fields such as well API number, operator name, lease name, production volumes (oil, gas, water), formation, field name, county, and geographic coordinates. Well log data is available through the KGS well log database, which contains digitized logs for many wells in the state. The data is valuable for decline curve analysis, production forecasting, reserve estimation, and machine learning applications in mature oil fields. Kansas represents a significant onshore production province in the mid-continent region of the United States."""
        },
        {
            "title": "Norne Field Benchmark Case",
            "source": "norne_dataset",
            "text": """The Norne benchmark case is a reservoir simulation dataset released by NTNU and Statoil (now Equinor) for research purposes. The Norne field is located in the Norwegian Sea. The benchmark includes a full reservoir simulation model with geological model, production history, and well data. It has been extensively used for history matching studies, uncertainty quantification, and optimization research. The model contains approximately 46,000 active cells, multiple reservoir zones, and various well control strategies."""
        },
        {
            "title": "SPE Comparative Solution Projects",
            "source": "spe_csp",
            "text": """The Society of Petroleum Engineers (SPE) has published a series of Comparative Solution Projects (CSPs) that serve as benchmark problems for reservoir simulation. These include: SPE 1 - a simple radial coning problem testing water and gas coning behavior; SPE 2 - radial composite problem for well testing; SPE 3 - testing gas cycling in a gas-condensate reservoir; SPE 5 - three-component, three-phase problem with gravity and capillary effects; SPE 9 - three-dimensional, three-phase, four-component problem testing black oil simulators; SPE 10 - model testing gas injection and compositional effects. These benchmark cases are widely used to validate reservoir simulators and compare different numerical approaches. They provide standardized input data, boundary conditions, and reference solutions that researchers can use to verify their simulation tools."""
        },
    ]

    for ds in datasets:
        records.append({
            "id": generate_id(ds['title']),
            "source": ds['source'],
            "title": ds['title'],
            "content_type": "dataset_documentation",
            "text": f"# {ds['title']}\n\n{clean_text(ds['text'])}",
            "scraped_at": datetime.now().isoformat()
        })

    # Also try to fetch live pages
    live_urls = [
        ("https://data.equinor.com/dataset/Volve", "volve_dataset"),
        ("https://github.com/bolgebrygg/Force-2020-Machine-Learning-competition", "force_2020"),
        ("https://www.kgs.ku.edu/PRS/petroDB.html", "kansas_gs"),
    ]
    for url, tag in live_urls:
        resp = safe_request(url)
        if resp:
            text = extract_text_from_html(resp.text, url)
            if text and len(text) > 100:
                records.append({
                    "id": generate_id(url), "source": tag, "title": tag,
                    "url": url, "content_type": "dataset_documentation",
                    "text": text, "scraped_at": datetime.now().isoformat()
                })
        polite_delay()

    logger.info(f"Dataset docs: {len(records)} records")
    return records

ds_records = collect_dataset_docs()
save_records(ds_records, "15_datasets.jsonl")
log_source("datasets", len(ds_records))


📊 Running total: 5,668 records from 11 sources


In [ ]:
# ============================================================
# CHECKPOINT: Verify all scraped data is saved to Drive
# ============================================================

import shutil

raw_files = list(RAW_DIR.glob("*.jsonl"))
total_raw_size = sum(f.stat().st_size for f in raw_files)

print(f"\n{'='*60}")
print(f"💾 SCRAPING CHECKPOINT — Saved to Google Drive")
print(f"{'='*60}")
print(f"   Location: {RAW_DIR}")
print(f"   Files:    {len(raw_files)}")
print(f"   Size:     {total_raw_size/(1024*1024):.1f} MB")
print(f"\n   Files on Drive:")
for f in sorted(raw_files):
    sz = f.stat().st_size / (1024*1024)
    print(f"      📄 {f.name} ({sz:.2f} MB)")

print(f"\n✅ All scraped data is safely on Google Drive.")
print(f"   If Colab disconnects, you can skip Phase 2 and resume from Phase 3.")


💾 SCRAPING CHECKPOINT — Saved to Google Drive
   Location: /content/drive/MyDrive/petroleum_corpus/raw
   Files:    18
   Size:     296.2 MB

   Files on Drive:
      📄 01_arxiv.jsonl (2.42 MB)
      📄 02_semantic_scholar.jsonl (38.51 MB)
      📄 03_openalex.jsonl (107.08 MB)
      📄 04_crossref.jsonl (35.26 MB)
      📄 05_doe_osti.jsonl (17.81 MB)
      📄 06_core.jsonl (16.36 MB)
      📄 07_doaj.jsonl (13.00 MB)
      📄 08_unpaywall.jsonl (2.26 MB)
      📄 09_petrowiki.jsonl (0.00 MB)
      📄 10_slb_glossary.jsonl (0.92 MB)
      📄 11_wikipedia.jsonl (1.43 MB)
      📄 12_eia.jsonl (47.41 MB)
      📄 13_epa.jsonl (0.05 MB)
      📄 13_jpt.jsonl (0.18 MB)
      📄 13_rigzone.jsonl (0.05 MB)
      📄 13_usgs.jsonl (0.23 MB)
      📄 14_onepetro.jsonl (13.18 MB)
      📄 15_datasets.jsonl (0.01 MB)

✅ All scraped data is safely on Google Drive.
   If Colab disconnects, you can skip Phase 2 and resume from Phase 3.


---
## PHASE 3: CONSOLIDATE & PREPARE SEED DATA FOR NVIDIA DATA DESIGNER

Merge all scraped sources, deduplicate, chunk, and prepare as a seed dataset
for NVIDIA NeMo Data Designer's synthetic data generation pipeline.

In [ ]:
# ============================================================
# INSTALL NVIDIA DATA DESIGNER
# ============================================================
!pip install data-designer -q

print("✅ NVIDIA NeMo Data Designer installed.")
print("   Docs: https://nvidia-nemo.github.io/DataDesigner/latest/")
print("   GitHub: https://github.com/NVIDIA-NeMo/DataDesigner")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.3/90.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.4/560.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.1/947.1 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7

In [ ]:
!pip install "pyarrow>=19.0.1,<20" --break-system-packages -q

In [ ]:
# ============================================================
# CONSOLIDATE ALL SCRAPED DATA INTO SEED DATASET
# ============================================================

def consolidate_for_data_designer():
    """
    Merge all scraped sources, deduplicate, chunk, and prepare
    a clean seed dataset for NVIDIA Data Designer.
    """
    import pandas as pd

    # ===== Load all records =====
    all_records = []
    source_counts = {}

    print("📂 Loading all JSONL files...")
    for f in sorted(RAW_DIR.glob("*.jsonl")):
        count = 0
        with jsonlines.open(f) as reader:
            for rec in reader:
                all_records.append(rec)
                count += 1
        source_counts[f.stem] = count
        print(f"   ✅ {f.stem}: {count:,}")

    print(f"\n📊 Total before dedup: {len(all_records):,}")

    # ===== Deduplicate =====
    seen, unique, dupes = set(), [], 0
    for rec in all_records:
        text = rec.get('text', '')
        if not text:
            continue
        h = hashlib.md5(text[:500].lower().encode()).hexdigest()
        if h not in seen:
            seen.add(h)
            unique.append(rec)
        else:
            dupes += 1

    # Filter short records
    filtered = [r for r in unique if len(r.get('text', '')) >= 100]
    print(f"🗑️  Removed {dupes:,} duplicates + {len(unique)-len(filtered):,} short records")
    print(f"✅ Unique records: {len(filtered):,}")

    # ===== Chunk for Data Designer =====
    print("\n🔪 Chunking text for Data Designer seed...")
    chunked = []
    chunk_size, overlap = 600, 80  # Words

    for r in filtered:
        text = r.get('text', '')
        words = text.split()
        title = r.get('title', r.get('term', ''))
        source = r.get('source', 'unknown')
        content_type = r.get('content_type', 'unknown')

        if len(words) <= chunk_size * 1.3:
            chunked.append({
                "text": text,
                "title": title,
                "source": source,
                "content_type": content_type,
                "word_count": len(words),
            })
        else:
            start = 0
            while start < len(words):
                end = min(start + chunk_size, len(words))
                chunk_text = ' '.join(words[start:end])
                chunked.append({
                    "text": chunk_text,
                    "title": title,
                    "source": source,
                    "content_type": content_type,
                    "word_count": len(chunk_text.split()),
                })
                start += chunk_size - overlap
                if end >= len(words):
                    break

    print(f"✅ Created {len(chunked):,} chunks from {len(filtered):,} records")

    # ===== Build seed DataFrame =====
    seed_df = pd.DataFrame(chunked)

    # Save as Parquet (preferred by Data Designer)
    seed_path = PROCESSED_DIR / "seed_corpus.parquet"
    seed_df.to_parquet(seed_path, index=False)

    # Also save as JSONL for reference
    jsonl_path = PROCESSED_DIR / "seed_corpus.jsonl"
    with jsonlines.open(jsonl_path, mode='w') as w:
        for rec in chunked:
            w.write(rec)

    # Save consolidated full records too
    save_records(filtered, "consolidated_corpus.jsonl", subdir="processed")

    # ===== Statistics =====
    stats = {
        "total_raw_records": len(all_records),
        "unique_records": len(filtered),
        "total_chunks": len(chunked),
        "total_words": int(seed_df['word_count'].sum()),
        "avg_chunk_words": int(seed_df['word_count'].mean()),
        "source_counts": source_counts,
        "content_types": dict(Counter(seed_df['content_type'])),
        "sources_in_seed": dict(Counter(seed_df['source'])),
    }
    with open(METADATA_DIR / "seed_statistics.json", 'w') as f:
        json.dump(stats, f, indent=2)

    print(f"\n{'='*60}")
    print(f"📊 SEED DATASET READY")
    print(f"{'='*60}")
    print(f"   Chunks:     {len(chunked):,}")
    print(f"   Words:      {stats['total_words']:,}")
    print(f"   Avg/chunk:  {stats['avg_chunk_words']}")
    print(f"   Saved to:   {seed_path}")
    print(f"\n   By content type:")
    for ct, cnt in sorted(stats['content_types'].items(), key=lambda x: -x[1]):
        print(f"      {ct:35s} {cnt:>6,}")

    return seed_df, stats

seed_df, seed_stats = consolidate_for_data_designer()
# Verify seed data is on Drive
print(f"\n💾 Seed data saved to Google Drive:")
print(f"   {PROCESSED_DIR / 'seed_corpus.parquet'}")
print(f"   {PROCESSED_DIR / 'seed_corpus.jsonl'}")
print(f"   {PROCESSED_DIR / 'consolidated_corpus.jsonl'}")

📂 Loading all JSONL files...
   ✅ 01_arxiv: 739
   ✅ 02_semantic_scholar: 10,082
   ✅ 03_openalex: 24,731
   ✅ 04_crossref: 9,937
   ✅ 05_doe_osti: 5,533
   ✅ 06_core: 565
   ✅ 07_doaj: 3,268
   ✅ 08_unpaywall: 235
   ✅ 09_petrowiki: 0
   ✅ 10_slb_glossary: 455
   ✅ 11_wikipedia: 1,089
   ✅ 12_eia: 442
   ✅ 13_epa: 16
   ✅ 13_jpt: 48
   ✅ 13_rigzone: 12
   ✅ 13_usgs: 62
   ✅ 14_onepetro: 2,971
   ✅ 15_datasets: 8

📊 Total before dedup: 60,193
🗑️  Removed 1,505 duplicates + 1 short records
✅ Unique records: 58,687

🔪 Chunking text for Data Designer seed...
✅ Created 66,905 chunks from 58,687 records

📊 SEED DATASET READY
   Chunks:     66,905
   Words:      20,519,557
   Avg/chunk:  306
   Saved to:   /content/drive/MyDrive/petroleum_corpus/processed/seed_corpus.parquet

   By content type:
      research_paper                      56,624
      technical_report                     5,529
      government_report                    2,306
      encyclopedia                         1,092
   

In [ ]:
# ============================================================
# CONSOLIDATE ALL SCRAPED DATA INTO SEED DATASET
# ============================================================
"""
def consolidate_for_data_designer():
    """
    Merge all scraped sources, deduplicate, chunk, and prepare
    a clean seed dataset for NVIDIA Data Designer.
    """
    import pandas as pd

    # ===== Load all records =====
    all_records = []
    source_counts = {}

    print("📂 Loading all JSONL files...")
    for f in sorted(RAW_DIR.glob("*.jsonl")):
        count = 0
        with jsonlines.open(f) as reader:
            for rec in reader:
                all_records.append(rec)
                count += 1
        source_counts[f.stem] = count
        print(f"   ✅ {f.stem}: {count:,}")

    print(f"\n📊 Total before dedup: {len(all_records):,}")

    # ===== Deduplicate =====
    seen, unique, dupes = set(), [], 0
    for rec in all_records:
        text = rec.get('text', '')
        if not text:
            continue
        h = hashlib.md5(text[:500].lower().encode()).hexdigest()
        if h not in seen:
            seen.add(h)
            unique.append(rec)
        else:
            dupes += 1

    # Filter short records
    filtered = [r for r in unique if len(r.get('text', '')) >= 100]
    print(f"🗑️  Removed {dupes:,} duplicates + {len(unique)-len(filtered):,} short records")
    print(f"✅ Unique records: {len(filtered):,}")

    # ===== Chunk for Data Designer =====
    print("\n🔪 Chunking text for Data Designer seed...")
    chunked = []
    chunk_size, overlap = 600, 80  # Words

    for r in filtered:
        text = r.get('text', '')
        words = text.split()
        title = r.get('title', r.get('term', ''))
        source = r.get('source', 'unknown')
        content_type = r.get('content_type', 'unknown')

        if len(words) <= chunk_size * 1.3:
            chunked.append({
                "text": text,
                "title": title,
                "source": source,
                "content_type": content_type,
                "word_count": len(words),
            })
        else:
            start = 0
            while start < len(words):
                end = min(start + chunk_size, len(words))
                chunk_text = ' '.join(words[start:end])
                chunked.append({
                    "text": chunk_text,
                    "title": title,
                    "source": source,
                    "content_type": content_type,
                    "word_count": len(chunk_text.split()),
                })
                start += chunk_size - overlap
                if end >= len(words):
                    break

    print(f"✅ Created {len(chunked):,} chunks from {len(filtered):,} records")

    # ===== Build seed DataFrame =====
    seed_df = pd.DataFrame(chunked)

    # Save as CSV (universally compatible, no PyArrow dependency)
    seed_path = PROCESSED_DIR / "seed_corpus.csv"
    seed_df.to_csv(seed_path, index=False)

    # Also save as JSONL for reference
    jsonl_path = PROCESSED_DIR / "seed_corpus.jsonl"
    with jsonlines.open(jsonl_path, mode='w') as w:
        for rec in chunked:
            w.write(rec)

    # Save consolidated full records too
    save_records(filtered, "consolidated_corpus.jsonl", subdir="processed")

    # ===== Statistics =====
    stats = {
        "total_raw_records": len(all_records),
        "unique_records": len(filtered),
        "total_chunks": len(chunked),
        "total_words": int(seed_df['word_count'].sum()),
        "avg_chunk_words": int(seed_df['word_count'].mean()),
        "source_counts": source_counts,
        "content_types": dict(Counter(seed_df['content_type'])),
        "sources_in_seed": dict(Counter(seed_df['source'])),
    }
    with open(METADATA_DIR / "seed_statistics.json", 'w') as f:
        json.dump(stats, f, indent=2)

    print(f"\n{'='*60}")
    print(f"📊 SEED DATASET READY")
    print(f"{'='*60}")
    print(f"   Chunks:     {len(chunked):,}")
    print(f"   Words:      {stats['total_words']:,}")
    print(f"   Avg/chunk:  {stats['avg_chunk_words']}")
    print(f"   Saved to:   {seed_path}")
    print(f"\n   By content type:")
    for ct, cnt in sorted(stats['content_types'].items(), key=lambda x: -x[1]):
        print(f"      {ct:35s} {cnt:>6,}")

    return seed_df, stats

seed_df, seed_stats = consolidate_for_data_designer()
print(f"\n💾 Seed data saved to Google Drive:")
print(f"   {PROCESSED_DIR / 'seed_corpus.csv'}")
print(f"   {PROCESSED_DIR / 'seed_corpus.jsonl'}")
print(f"   {PROCESSED_DIR / 'consolidated_corpus.jsonl'}")

IndentationError: unexpected indent (2769403267.py, line 7)

---
## PHASE 4: NVIDIA DATA DESIGNER — SYNTHETIC INSTRUCTION-RESPONSE GENERATION

This phase uses **NVIDIA NeMo Data Designer** to generate high-quality
instruction-response pairs from the scraped petroleum engineering corpus.

Data Designer provides:
- **Seed dataset integration** — scraped text feeds the LLM context
- **Statistical samplers** — ensure diversity across content types, categories, and complexity
- **LLM-powered generation** — instruction + response generation with Jinja2 templating
- **LLM-as-Judge scoring** — automated quality evaluation
- **Validation** — schema and quality checks on generated data
- **Batch parallelism** — efficient generation at scale

### Requirements
You need **one** of these API keys:
- **NVIDIA Build** (recommended): Free at https://build.nvidia.com
- **OpenAI**: https://platform.openai.com
- **OpenRouter**: https://openrouter.ai

In [ ]:
# ============================================================
# CONFIGURE API KEY FOR DATA DESIGNER
# ============================================================
# Set ONE of the following. NVIDIA Build API is recommended (free tier available).
# Get your key at: https://build.nvidia.com

import os

from google.colab import userdata
NVIDIA_API_KEY = userdata.get('NVIDIA_API_KEY')

# Export the key to environment variables so Data Designer can find it
if NVIDIA_API_KEY:
    os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY

# Option 2: OpenAI API
# os.environ["OPENAI_API_KEY"] = ""  # ← Paste your OpenAI key here

# Option 3: OpenRouter API
# os.environ["OPENROUTER_API_KEY"] = ""  # ← Paste your OpenRouter key here

# Verify
has_nvidia = bool(os.environ.get('NVIDIA_API_KEY'))
has_openai = bool(os.environ.get("OPENAI_API_KEY"))
has_openrouter = bool(os.environ.get("OPENROUTER_API_KEY"))

if has_nvidia:
    print("✅ Using NVIDIA Build API")
elif has_openai:
    print("✅ Using OpenAI API")
elif has_openrouter:
    print("✅ Using OpenRouter API")
else:
    print("☀️  No API key set! Please set one of the keys above.")
    print("   NVIDIA Build API is free: https://build.nvidia.com")

✅ Using NVIDIA Build API


In [ ]:
# ============================================================
# INITIALIZE NVIDIA DATA DESIGNER
# ============================================================

import data_designer.config as dd
from data_designer.interface import DataDesigner

# Initialize Data Designer — it auto-detects available API keys
data_designer = DataDesigner()

print("✅ NVIDIA Data Designer initialized.")
print("   Available model providers will be auto-detected from environment.")

✅ NVIDIA Data Designer initialized.
   Available model providers will be auto-detected from environment.


### 4.1 — Pipeline 1: Generate Instruction-Response Pairs from Seed Corpus

This is the primary pipeline. It uses the scraped petroleum engineering text as seed data
and generates diverse instruction-response pairs using Data Designer's column system.

In [ ]:
# ============================================================
# PIPELINE 1: INSTRUCTION-RESPONSE PAIRS FROM SEED DATA
# ============================================================

config_ir = dd.DataDesignerConfigBuilder()

# ─── Sampler 1: Instruction category (controls diversity) ───
config_ir.add_column(
    dd.SamplerColumnConfig(
        name="instruction_category",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "concept_explanation",
                "calculation_problem",
                "procedure_description",
                "troubleshooting",
                "comparison_analysis",
                "design_recommendation",
                "data_interpretation",
                "best_practices",
                "safety_compliance",
                "case_study_analysis",
            ],
        ),
    )
)

# ─── Sampler 2: Complexity level ───
config_ir.add_column(
    dd.SamplerColumnConfig(
        name="complexity_level",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["beginner", "intermediate", "advanced", "expert"],
        ),
    )
)

# ─── Sampler 3: Petroleum subdomain ───
config_ir.add_column(
    dd.SamplerColumnConfig(
        name="subdomain",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "reservoir_engineering",
                "drilling_engineering",
                "production_engineering",
                "completion_stimulation",
                "formation_evaluation",
                "flow_assurance",
                "enhanced_oil_recovery",
                "offshore_subsea",
                "process_facilities",
                "petroleum_geoscience",
                "well_integrity_safety",
                "digital_oilfield_AI",
            ],
        ),
    )
)

# ─── LLM Column 1: Generate the instruction/question ───
config_ir.add_column(
    dd.LLMTextColumnConfig(
        name="instruction",
        model_alias="nvidia-text",
        system_prompt=(
            "You are an expert petroleum engineering educator and technical writer. "
            "You create precise, technically accurate questions and instructions "
            "for training AI models on oil and gas domain knowledge."
        ),
        prompt=(
            "Generate a single, specific petroleum engineering question or instruction "
            "that matches these parameters:\n\n"
            "Category: {{ instruction_category }}\n"
            "Complexity: {{ complexity_level }}\n"
            "Subdomain: {{ subdomain }}\n\n"
            "Guidelines:\n"
            "- The question should be self-contained and specific\n"
            "- For 'calculation_problem': include realistic numerical parameters\n"
            "- For 'troubleshooting': describe a specific operational scenario\n"
            "- For 'comparison_analysis': name specific methods or technologies\n"
            "- For 'design_recommendation': provide realistic well/field conditions\n"
            "- Match the complexity level (beginner=basic concepts, expert=advanced analysis)\n\n"
            "Output ONLY the question/instruction text, nothing else."
        ),
    )
)

# ─── LLM Column 2: Generate the expert response ───
config_ir.add_column(
    dd.LLMTextColumnConfig(
        name="response",
        model_alias="nvidia-text",
        system_prompt=(
            "You are a senior petroleum engineer with 25+ years of field and academic experience. "
            "You provide detailed, technically accurate responses that include specific values, "
            "equations, industry standards (API, SPE, ISO), and practical field considerations. "
            "Use proper engineering units (both field and SI where appropriate)."
        ),
        prompt=(
            "Provide a comprehensive, technically accurate response to the following "
            "petroleum engineering question.\n\n"
            "Question: {{ instruction }}\n\n"
            "Requirements:\n"
            "- Complexity target: {{ complexity_level }}\n"
            "- Include specific technical details, equations, and industry standards\n"
            "- Reference real methods, tools, or software where relevant\n"
            "- Include practical considerations and common field challenges\n"
            "- For calculations, show the full working with realistic values\n"
            "- Aim for 200-500 words depending on complexity\n\n"
            "Provide the response directly without preamble."
        ),
    )
)

print("✅ Pipeline 1 configured: Instruction-Response generation")
print("   Columns: instruction_category, complexity_level, subdomain → instruction → response")
print("   Model: nvidia-text (default backend model)")

✅ Pipeline 1 configured: Instruction-Response generation
   Columns: instruction_category, complexity_level, subdomain → instruction → response
   Model: nvidia-text (default backend model)


In [ ]:
# ============================================================
# PREVIEW PIPELINE 1 — Validate before full generation
# ============================================================

print("🔍 Generating preview (10 sample records)...")
preview_ir = data_designer.preview(config_builder=config_ir)

print("\n" + "="*70)
print("📋 PIPELINE 1 PREVIEW — Instruction-Response Pairs")
print("="*70)

# Display sample records
preview_ir.display_sample_record()

# Show the preview DataFrame
print("\n📊 Preview DataFrame:")
df_preview = preview_ir.dataset
print(df_preview[['instruction_category', 'complexity_level', 'subdomain']].value_counts().head(10))
print(f"\n📏 Average instruction length: {df_preview['instruction'].str.len().mean():.0f} chars")
print(f"📏 Average response length:    {df_preview['response'].str.len().mean():.0f} chars")

[19:59:06] [INFO] 🧐 Preview generation in progress
[19:59:06] [INFO] ✅ Validation passed
[19:59:06] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[19:59:06] [INFO] 🩺 Running health checks for models...
[19:59:06] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'nvidia-text'...


🔍 Generating preview (10 sample records)...


[19:59:08] [INFO]   |-- ✅ Passed!
[19:59:08] [INFO] 🎲 Preparing samplers to generate 10 records across 3 columns
[19:59:08] [INFO] 📝 llm-text model config for column 'instruction'
[19:59:08] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[19:59:08] [INFO]   |-- model alias: 'nvidia-text'
[19:59:08] [INFO]   |-- model provider: 'nvidia'
[19:59:08] [INFO]   |-- inference parameters:
[19:59:08] [INFO]   |  |-- generation_type=chat-completion
[19:59:08] [INFO]   |  |-- max_parallel_requests=4
[19:59:08] [INFO]   |  |-- temperature=1.00
[19:59:08] [INFO]   |  |-- top_p=1.00
[19:59:08] [INFO] ⚡️ Processing llm-text column 'instruction' with 4 concurrent workers
[19:59:08] [INFO] ⏱️ llm-text column 'instruction' will report progress after each record
[19:59:09] [INFO]   |-- 🥚 llm-text column 'instruction' progress: 1/10 (10%) complete, 1 ok, 0 failed, 1.08 rec/s, eta 8.4s
[19:59:09] [INFO]   |-- 🥚 llm-text column 'instruction' progress: 2/10 (20%) complete, 2 ok, 0 failed, 1.87 rec/s, e


📋 PIPELINE 1 PREVIEW — Instruction-Response Pairs


                                              Generated Columns                                               
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                 ┃ Value                                                                               ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ instruction_category │ troubleshooting                                                                     │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ complexity_level     │ beginner                                                                            │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ subdomain            │ process_facilities                                                                  │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ instruction          │ During routine operation of a surface gas‑lift unit in a processing facility, the   │
│                      │ measured gas‑lift injection pressure remains constant but the produced fluid flow   │
│                      │ rate drops by 30 % over two hours while all equipment parameters are within design  │
│                      │ limits. Identify the most likely operational cause and describe the immediate       │
│                      │ corrective steps to restore normal flow.                                            │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ response             │ **Likely cause** – A drop in produced‑fluid rate while injection pressure is        │
│                      │ unchanged most often signals that the effective lift pressure in the tubing has     │
│                      │ risen. In practice this is usually the result of a **partial blockage or            │
│                      │ water‑breakthrough in the gas‑lift valve (or downstream tubing)** that reduces the  │
│                      │ amount of gas actually entering the well. The valve may be sticking, fouled with    │
│                      │ scale/hydrates, or the opening pressure may have shifted upward after a change in   │
│                      │ fluid composition (e.g., higher water cut). Because the injector pressure is held   │
│                      │ constant, the pressure differential that drives the lift (                          │
│                      │ \(P_{\text{inj}}-P_{\text{stat}}\) ) shrinks, and the well “un‑lifts,” cutting flow │
│                      │ by roughly the same percentage.                                                     │
│                      │                                                                                     │
│                      │ ---                                                                                 │
│                      │                                                                                     │
│                      │ ### Simple pressure‑flow calculation (field example)                                │
│                      │                                                                                     │
│                      │ Assume a typical gas‑lift well:                                                     │
│                      │                                                                                     │
│                      │ * Injection pressure, \(P_{\text{inj}} = 1 500\) psi (kept constant)                │
│                      │ * Initial static (bottom‑hole) pressure, \(P_{\text{stat,0}} = 800\) psi            │
│                      │ * Initial production, \(Q_0 = 1 000\) bbl day⁻¹                                     │
│   


📊 Preview DataFrame:
instruction_category  complexity_level  subdomain             
best_practices        intermediate      production_engineering    1
calculation_problem   advanced          offshore_subsea           1
                      expert            formation_evaluation      1
case_study_analysis   beginner          offshore_subsea           1
comparison_analysis   advanced          well_integrity_safety     1
concept_explanation   beginner          process_facilities        1
data_interpretation   expert            completion_stimulation    1
troubleshooting       beginner          process_facilities        1
                      expert            digital_oilfield_AI       1
                                        flow_assurance            1
Name: count, dtype: int64

📏 Average instruction length: 498 chars
📏 Average response length:    3856 chars


In [ ]:
# ============================================================
# FULL GENERATION — Pipeline 1: Instruction-Response Pairs
# ============================================================
# Adjust num_records based on your budget and needs.
# Each record costs ~2 LLM calls (instruction + response).
#
# Recommended targets:
#   - Quick test:   500 records
#   - Medium run:   5,000 records
#   - Full target:  15,000 records

NUM_IR_RECORDS = 20000  # ← Adjust this

print(f"🚀 Generating {NUM_IR_RECORDS:,} instruction-response pairs...")
print(f"   This will make ~{NUM_IR_RECORDS * 2:,} LLM API calls.")
print(f"   Estimated time: {NUM_IR_RECORDS * 2 / 60:.0f}-{NUM_IR_RECORDS * 4 / 60:.0f} minutes")
print()

results_ir = data_designer.create(
    config_builder=config_ir,
    num_records=NUM_IR_RECORDS,
    dataset_name="petroleum_instruction_response",
)

df_ir = results_ir.dataset

print(f"\n✅ Generated {len(df_ir):,} instruction-response pairs")
print(f"   Columns: {list(df_ir.columns)}")

# Save intermediate results
ir_path = AUGMENTED_DIR / "pipeline1_instruction_response.parquet"
df_ir.to_parquet(ir_path, index=False)
print(f"💾 Saved to: {ir_path}")
# Also save as JSONL to Drive for safety
ir_jsonl_path = AUGMENTED_DIR / "pipeline1_instruction_response.jsonl"
with jsonlines.open(ir_jsonl_path, mode='w') as w:
    for _, row in df_ir.iterrows():
        w.write(row.to_dict())
print(f"💾 Also saved JSONL backup: {ir_jsonl_path}")
print(f"💾 All Pipeline 1 data is on Google Drive.")


[20:17:46] [INFO] 🎨 Creating Data Designer dataset
[20:17:46] [INFO] 📂 Dataset path '/content/artifacts/petroleum_instruction_response' already exists. Dataset from this session
		     will be saved to '/content/artifacts/petroleum_instruction_response_03-17-2026_201746' instead.
[20:17:46] [INFO] ✅ Validation passed
[20:17:46] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[20:17:46] [INFO] 🩺 Running health checks for models...
[20:17:46] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'nvidia-text'...


🚀 Generating 20,000 instruction-response pairs...
   This will make ~40,000 LLM API calls.
   Estimated time: 667-1333 minutes



[20:17:46] [INFO]   |-- ✅ Passed!
[20:17:46] [INFO] ⏳ Processing batch 1 of 20
[20:17:46] [INFO] 🎲 Preparing samplers to generate 1000 records across 3 columns
[20:17:46] [INFO] 📝 llm-text model config for column 'instruction'
[20:17:46] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[20:17:46] [INFO]   |-- model alias: 'nvidia-text'
[20:17:46] [INFO]   |-- model provider: 'nvidia'
[20:17:46] [INFO]   |-- inference parameters:
[20:17:46] [INFO]   |  |-- generation_type=chat-completion
[20:17:46] [INFO]   |  |-- max_parallel_requests=4
[20:17:46] [INFO]   |  |-- temperature=1.00
[20:17:46] [INFO]   |  |-- top_p=1.00
[20:17:46] [INFO] ⚡️ Processing llm-text column 'instruction' with 4 concurrent workers
[20:17:46] [INFO] ⏱️ llm-text column 'instruction' will report progress every 100 records
[20:18:47] [INFO]   |-- 🌧️ llm-text column 'instruction' progress: 100/1000 (10%) complete, 100 ok, 0 failed, 1.64 rec/s, eta 550.0s
[20:19:50] [INFO]   |-- 🌧️ llm-text column 'instruction' pro

### 4.2 — Pipeline 2: Seed-Grounded QA from Scraped Corpus

This pipeline uses the actual scraped text as context to generate
QA pairs grounded in real petroleum engineering literature.

In [ ]:
# ============================================================
# PIPELINE 2: SEED-GROUNDED QA FROM SCRAPED CORPUS
# ============================================================

# Load seed data from the consolidated corpus
seed_path = PROCESSED_DIR / "seed_corpus.parquet"

config_seed = dd.DataDesignerConfigBuilder()

# ─── Load seed dataset ───
# The seed dataset columns (text, title, source, content_type) become
# available as Jinja2 variables in all LLM prompts.
config_seed.with_seed_dataset(
    file_path=str(seed_path),
    sampling_strategy="shuffle",
)

# ─── Sampler: QA generation style ───
config_seed.add_column(
    dd.SamplerColumnConfig(
        name="qa_style",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "summarize_key_points",
                "explain_methodology",
                "extract_technical_details",
                "identify_applications",
                "compare_with_alternatives",
                "troubleshoot_scenario",
                "design_based_on_context",
                "evaluate_limitations",
            ],
        ),
    )
)

# ─── LLM Column 1: Generate question grounded in seed text ───
config_seed.add_column(
    dd.LLMTextColumnConfig(
        name="instruction",
        model_alias="nvidia-text",
        system_prompt=(
            "You are an expert petroleum engineering educator creating training data "
            "for a domain-specific AI model. Generate questions that test deep "
            "understanding of petroleum engineering concepts."
        ),
        prompt=(
            "Based on the following petroleum engineering text, generate a specific, "
            "detailed question that can be fully answered using the information provided.\n\n"
            "Source: {{ source }} | Type: {{ content_type }}\n"
            "Title: {{ title }}\n\n"
            "Text:\n{{ text }}\n\n"
            "Question style: {{ qa_style }}\n\n"
            "Generate ONLY the question. Make it specific and technical."
        ),
    )
)

# ─── LLM Column 2: Generate answer grounded in seed text ───
config_seed.add_column(
    dd.LLMTextColumnConfig(
        name="response",
        model_alias="nvidia-text",
        system_prompt=(
            "You are a senior petroleum engineer providing precise, detailed answers. "
            "Base your answer on the provided context but expand with your domain expertise. "
            "Include specific technical details, equations, and practical considerations."
        ),
        prompt=(
            "Answer the following petroleum engineering question using the provided context. "
            "Expand beyond the context where your domain expertise allows, but stay accurate.\n\n"
            "Context from {{ source }}:\n{{ text }}\n\n"
            "Question: {{ instruction }}\n\n"
            "Provide a comprehensive, technically accurate answer (200-500 words)."
        ),
    )
)

print("✅ Pipeline 2 configured: Seed-grounded QA generation")
print(f"   Seed dataset: {seed_path} ({len(seed_df):,} chunks)")
print("   Columns: seed[text,title,source,content_type] + qa_style → instruction → response")

In [ ]:
# ============================================================
# PREVIEW PIPELINE 2
# ============================================================

print("🔍 Generating preview for seed-grounded QA...")
preview_seed = data_designer.preview(config_builder=config_seed)

print("\n" + "="*70)
print("📋 PIPELINE 2 PREVIEW — Seed-Grounded QA")
print("="*70)
preview_seed.display_sample_record()

df_prev2 = preview_seed.dataset
print(f"\n📏 Average instruction length: {df_prev2['instruction'].str.len().mean():.0f} chars")
print(f"📏 Average response length:    {df_prev2['response'].str.len().mean():.0f} chars")
print(f"📊 Seed sources used: {df_prev2['source'].value_counts().to_dict()}")

In [ ]:
# ============================================================
# FULL GENERATION — Pipeline 2: Seed-Grounded QA
# ============================================================

NUM_SEED_RECORDS = 20000  # ← Adjust this

print(f"🚀 Generating {NUM_SEED_RECORDS:,} seed-grounded QA pairs...")
print(f"   Estimated time: {NUM_SEED_RECORDS * 2 / 60:.0f}-{NUM_SEED_RECORDS * 4 / 60:.0f} minutes")
print()

results_seed = data_designer.create(
    config_builder=config_seed,
    num_records=NUM_SEED_RECORDS,
    dataset_name="petroleum_seed_qa",
)

df_seed_qa = results_seed.dataset

print(f"\n✅ Generated {len(df_seed_qa):,} seed-grounded QA pairs")

# Save intermediate
seed_qa_path = AUGMENTED_DIR / "pipeline2_seed_grounded_qa.parquet"
df_seed_qa.to_parquet(seed_qa_path, index=False)
print(f"💾 Saved to: {seed_qa_path}")
# Also save as JSONL to Drive for safety
seed_jsonl_path = AUGMENTED_DIR / "pipeline2_seed_grounded_qa.jsonl"
with jsonlines.open(seed_jsonl_path, mode='w') as w:
    for _, row in df_seed_qa.iterrows():
        w.write(row.to_dict())
print(f"💾 Also saved JSONL backup: {seed_jsonl_path}")
print(f"💾 All Pipeline 2 data is on Google Drive.")


### 4.3 — Pipeline 3: Quality Scoring with LLM-as-Judge

Use Data Designer's `LLMJudgeColumnConfig` to evaluate the quality of generated pairs.

In [ ]:
# ============================================================
# PIPELINE 3: QUALITY SCORING
# ============================================================
# Merge both pipeline outputs, then score a sample for quality.

import pandas as pd

# Combine all generated data
all_generated = []

# Pipeline 1 results
if 'df_ir' in dir() and df_ir is not None and len(df_ir) > 0:
    df_ir_clean = df_ir[['instruction', 'response', 'instruction_category',
                         'complexity_level', 'subdomain']].copy()
    df_ir_clean['pipeline'] = 'knowledge_generation'
    df_ir_clean['source_type'] = 'synthetic'
    all_generated.append(df_ir_clean)
    print(f"✅ Pipeline 1: {len(df_ir_clean):,} records")

# Pipeline 2 results
if 'df_seed_qa' in dir() and df_seed_qa is not None and len(df_seed_qa) > 0:
    df_seed_clean = df_seed_qa[['instruction', 'response']].copy()
    if 'qa_style' in df_seed_qa.columns:
        df_seed_clean['instruction_category'] = df_seed_qa['qa_style']
    if 'content_type' in df_seed_qa.columns:
        df_seed_clean['source_type'] = df_seed_qa['content_type']
    else:
        df_seed_clean['source_type'] = 'seed_grounded'
    df_seed_clean['pipeline'] = 'seed_grounded_qa'
    all_generated.append(df_seed_clean)
    print(f"✅ Pipeline 2: {len(df_seed_clean):,} records")

df_all = pd.concat(all_generated, ignore_index=True)
print(f"\n📊 Total combined records: {len(df_all):,}")

# ─── Score a sample with LLM Judge ───
# We score a random sample to avoid scoring the entire dataset (expensive)
JUDGE_SAMPLE_SIZE = min(500, len(df_all))
df_sample = df_all.sample(n=JUDGE_SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# Save as seed for judge pipeline
judge_seed_path = PROCESSED_DIR / "judge_sample.parquet"
df_sample.to_parquet(judge_seed_path, index=False)

# Configure judge pipeline
config_judge = dd.DataDesignerConfigBuilder()

config_judge.with_seed_dataset(
    file_path=str(judge_seed_path),
    sampling_strategy="sequential",
)

config_judge.add_column(
    dd.LLMJudgeColumnConfig(
        name="quality_score",
        model_alias="nvidia-text",
        prompt=(
            "Evaluate this petroleum engineering instruction-response pair:\n\n"
            "INSTRUCTION: {{ instruction }}\n\n"
            "RESPONSE: {{ response }}"
        ),
        scores=[
            dd.Score(
                name="technical_accuracy",
                description="Is the response technically accurate for petroleum engineering?",
                options={
                    "4": "Completely accurate with proper terminology and values",
                    "3": "Mostly accurate with minor issues",
                    "2": "Partially accurate with some errors",
                    "1": "Mostly inaccurate",
                    "0": "Completely inaccurate or irrelevant"
                }
            ),
            dd.Score(
                name="completeness",
                description="Does the response fully address the instruction?",
                options={
                    "4": "Comprehensive with all key aspects covered",
                    "3": "Good coverage of most aspects",
                    "2": "Partially addresses the question",
                    "1": "Mostly incomplete",
                    "0": "Does not address the question"
                }
            ),
            dd.Score(
                name="usefulness",
                description="Would this be useful for training a petroleum engineering AI?",
                options={
                    "4": "Excellent training example with specific details",
                    "3": "Good training example",
                    "2": "Acceptable but could be better",
                    "1": "Low quality for training",
                    "0": "Not useful for training"
                }
            ),
        ],
    )
)

print(f"✅ Judge pipeline configured for {JUDGE_SAMPLE_SIZE} sample records")
print("   Scoring: technical_accuracy, completeness, usefulness (0-4 each)")

In [ ]:
# ============================================================
# RUN QUALITY SCORING
# ============================================================

print(f"⚖️ Scoring {JUDGE_SAMPLE_SIZE} records for quality...")
judge_results = data_designer.create(
    config_builder=config_judge,
    num_records=JUDGE_SAMPLE_SIZE,
    dataset_name="petroleum_quality_scores",
)

df_scored = judge_results.dataset

# Extract numeric scores
for score_col in ['quality_score_technical_accuracy', 'quality_score_completeness', 'quality_score_usefulness']:
    if score_col in df_scored.columns:
        df_scored[score_col] = pd.to_numeric(df_scored[score_col], errors='coerce')

# Calculate composite score
score_cols = [c for c in df_scored.columns if c.startswith('quality_score_') and df_scored[c].dtype in ['float64', 'int64']]
if score_cols:
    df_scored['composite_score'] = df_scored[score_cols].mean(axis=1)

    print(f"\n{'='*60}")
    print("📊 QUALITY SCORE DISTRIBUTION")
    print(f"{'='*60}")
    for col in score_cols:
        print(f"   {col:45s}  mean={df_scored[col].mean():.2f}  std={df_scored[col].std():.2f}")
    print(f"   {'composite_score':45s}  mean={df_scored['composite_score'].mean():.2f}  std={df_scored['composite_score'].std():.2f}")

    # Determine quality threshold
    high_quality = df_scored[df_scored['composite_score'] >= 3.0]
    print(f"\n   High quality (≥3.0):  {len(high_quality):,} / {len(df_scored):,} ({100*len(high_quality)/len(df_scored):.1f}%)")

    # Save scored sample
    scored_path = AUGMENTED_DIR / "quality_scored_sample.parquet"
    df_scored.to_parquet(scored_path, index=False)
    print(f"\n💾 Scored sample saved to: {scored_path}")
else:
    print("⚠️ Could not extract numeric quality scores. Check judge output format.")

---
## PHASE 5: FINAL EXPORT — Merge, Filter, Format for Fine-Tuning

In [ ]:
# ============================================================
# FINAL EXPORT — Formatted for LLM Fine-Tuning
# ============================================================

import pandas as pd

# ===== Load all generated data =====
print("📂 Loading all generated data...")
all_dfs = []

for pfile in AUGMENTED_DIR.glob("pipeline*.parquet"):
    df = pd.read_parquet(pfile)
    print(f"   ✅ {pfile.name}: {len(df):,} records")
    all_dfs.append(df)

if not all_dfs:
    print("⚠️ No generated data found. Run the generation pipelines first.")
else:
    df_final = pd.concat(all_dfs, ignore_index=True)
    print(f"\n📊 Total records before filtering: {len(df_final):,}")

    # ===== Quality Filtering =====
    # Remove records with empty instruction or response
    df_final = df_final.dropna(subset=['instruction', 'response'])
    df_final = df_final[
        (df_final['instruction'].str.len() > 20) &
        (df_final['response'].str.len() > 50)
    ].copy()

    # Deduplicate by instruction
    df_final = df_final.drop_duplicates(subset=['instruction'], keep='first')

    print(f"   After quality filtering: {len(df_final):,}")

    # ===== Export Format 1: Alpaca-style JSONL =====
    # {instruction, input, output} — standard for fine-tuning
    alpaca_path = PROCESSED_DIR / "final_alpaca_format.jsonl"
    with jsonlines.open(alpaca_path, mode='w') as w:
        for _, row in df_final.iterrows():
            w.write({
                "instruction": row['instruction'],
                "input": "",
                "output": row['response'],
            })
    print(f"\n💾 Format 1 (Alpaca):      {alpaca_path}")

    # ===== Export Format 2: ShareGPT-style JSONL =====
    # {conversations: [{from, value}]} — used by many fine-tuning frameworks
    sharegpt_path = PROCESSED_DIR / "final_sharegpt_format.jsonl"
    with jsonlines.open(sharegpt_path, mode='w') as w:
        for _, row in df_final.iterrows():
            w.write({
                "conversations": [
                    {"from": "human", "value": row['instruction']},
                    {"from": "gpt", "value": row['response']},
                ]
            })
    print(f"💾 Format 2 (ShareGPT):    {sharegpt_path}")

    # ===== Export Format 3: Full metadata Parquet =====
    full_path = PROCESSED_DIR / "final_full_dataset.parquet"
    df_final.to_parquet(full_path, index=False)
    print(f"💾 Format 3 (Full Parquet): {full_path}")

    # ===== Export Format 4: CSV summary =====
    summary = df_final[['instruction', 'response']].copy()
    summary['instruction_len'] = summary['instruction'].str.len()
    summary['response_len'] = summary['response'].str.len()
    summary['instruction_preview'] = summary['instruction'].str[:100]
    summary['response_preview'] = summary['response'].str[:100]
    summary[['instruction_preview', 'response_preview', 'instruction_len', 'response_len']].to_csv(
        METADATA_DIR / "final_dataset_summary.csv", index=False
    )

    # ===== Final Statistics =====
    word_counts_i = df_final['instruction'].str.split().str.len()
    word_counts_r = df_final['response'].str.split().str.len()

    final_stats = {
        "total_instruction_response_pairs": len(df_final),
        "total_instruction_words": int(word_counts_i.sum()),
        "total_response_words": int(word_counts_r.sum()),
        "avg_instruction_words": int(word_counts_i.mean()),
        "avg_response_words": int(word_counts_r.mean()),
        "pipeline_breakdown": dict(df_final.get('pipeline', pd.Series(dtype=str)).value_counts()) if 'pipeline' in df_final.columns else {},
        "generated_at": datetime.now().isoformat(),
        "generated_with": "NVIDIA NeMo Data Designer",
    }
    with open(METADATA_DIR / "final_statistics.json", 'w') as f:
        json.dump(final_stats, f, indent=2, default=str)

    print(f"\n{'='*70}")
    print("🏆 FINAL DATASET STATISTICS")
    print(f"{'='*70}")
    print(f"📊 Total instruction-response pairs: {final_stats['total_instruction_response_pairs']:,}")
    print(f"📝 Total instruction words:          {final_stats['total_instruction_words']:,}")
    print(f"📝 Total response words:             {final_stats['total_response_words']:,}")
    print(f"📐 Avg instruction words:            {final_stats['avg_instruction_words']}")
    print(f"📐 Avg response words:               {final_stats['avg_response_words']}")
    if final_stats['pipeline_breakdown']:
        print(f"\n   By pipeline:")
        for p, cnt in final_stats['pipeline_breakdown'].items():
            print(f"      {p:35s} {cnt:>6,}")
    # ===== Verify everything is on Google Drive =====
    print(f"\n{'='*70}")
    print("💾 ALL FILES SAVED TO GOOGLE DRIVE")
    print(f"{'='*70}")
    print(f"📂 Drive location: {OUTPUT_DIR}")
    total_drive_size = sum(f.stat().st_size for f in OUTPUT_DIR.rglob('*') if f.is_file())
    print(f"💾 Total size on Drive: {total_drive_size/(1024*1024):.1f} MB")
    print(f"\n   Your data is safe even if Colab disconnects!")
    print(f"   Path: Google Drive > My Drive > petroleum_corpus")


In [ ]:
# ============================================================
# FINAL OUTPUT STRUCTURE
# ============================================================

print("\n📁 Output Directory:")
print("=" * 60)
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(str(OUTPUT_DIR), '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📂 {os.path.basename(root)}/")
    for f in sorted(files):
        fp = Path(root) / f
        size = fp.stat().st_size / (1024*1024)
        print(f"{indent}  📄 {f} ({size:.2f} MB)")

total = sum(f.stat().st_size for f in OUTPUT_DIR.rglob('*') if f.is_file())
print(f"\n💾 Total size: {total/(1024*1024):.1f} MB")

print(f"""
{'='*70}
✅ DATASET GENERATION COMPLETE — NVIDIA Data Designer
   ALL FILES SAVED TO GOOGLE DRIVE
{'='*70}

📂 Google Drive location:
   My Drive > petroleum_corpus

📋 Output files (all on Drive):

  FOR FINE-TUNING:
  ├── processed/final_alpaca_format.jsonl    (Alpaca: instruction/input/output)
  ├── processed/final_sharegpt_format.jsonl  (ShareGPT: conversations format)
  └── processed/final_full_dataset.parquet   (Full metadata incl. scores)

  INTERMEDIATE (on Drive):
  ├── augmented/pipeline1_instruction_response.parquet  (knowledge-based QA)
  ├── augmented/pipeline1_instruction_response.jsonl    (JSONL backup)
  ├── augmented/pipeline2_seed_grounded_qa.parquet      (corpus-grounded QA)
  ├── augmented/pipeline2_seed_grounded_qa.jsonl        (JSONL backup)
  └── augmented/quality_scored_sample.parquet            (LLM judge scores)

  RAW CORPUS (on Drive):
  ├── raw/*.jsonl                              (all scraped source files)
  ├── processed/seed_corpus.parquet            (chunked seed for Data Designer)
  └── processed/consolidated_corpus.jsonl       (full deduplicated corpus)

  METADATA (on Drive):
  ├── metadata/seed_statistics.json
  ├── metadata/final_statistics.json
  └── metadata/final_dataset_summary.csv

📋 Next steps:
  1. Files are already on Drive — download or access from any device
  2. Load final_alpaca_format.jsonl or final_sharegpt_format.jsonl
  3. Fine-tune with Unsloth + Llama 3.1 / Mistral / Qwen
  4. Use the quality scores to further filter if needed
  5. Evaluate with domain-specific benchmarks

💡 TIP: If Colab disconnected during scraping, just remount Drive
   and skip to Phase 3 — your raw/ files are already saved!
""")

---
## 📌 NOTES

### API Keys Required
| Service | URL | Purpose |
|---------|-----|--------|
| **NVIDIA Build** (recommended) | https://build.nvidia.com | Data Designer LLM generation |
| EIA (optional) | https://www.eia.gov/opendata/register.php | Production data API access |
| CORE (optional) | https://core.ac.uk/services/api | Full-text open-access papers |
| Email (OpenAlex/Crossref) | Just set POLITE_EMAIL | 10x higher rate limits |

### NVIDIA Data Designer Features Used
| Feature | Purpose |
|---------|--------|
| `SamplerColumnConfig` (Category) | Diverse instruction categories, complexity levels, subdomains |
| `LLMTextColumnConfig` | Instruction generation, response generation |
| `LLMJudgeColumnConfig` | Quality scoring (technical accuracy, completeness, usefulness) |
| `with_seed_dataset()` | Feed scraped corpus as context for grounded generation |
| `preview()` | Validate pipeline output before full-scale generation |
| `create()` | Full-scale parallel generation with batching |

### Scaling the Dataset
1. Increase `NUM_IR_RECORDS` (Pipeline 1) for more synthetic instruction-response pairs
2. Increase `NUM_SEED_RECORDS` (Pipeline 2) for more corpus-grounded QA pairs
3. Add more scrapers or increase `MAX_ITEMS_PER_QUERY` for a larger seed corpus
4. Add a CORE API key to get full-text papers in the seed
5. Use `push_to_hub()` to publish the dataset to Hugging Face Hub

### Runtime Estimates
- **Scraping (Phase 2):** ~3-5 hours (unchanged)
- **Seed preparation (Phase 3):** ~2-5 minutes
- **Data Designer generation (Phase 4):**
  - Preview: ~30 seconds per pipeline
  - 5,000 records: ~30-60 minutes
  - 15,000 records: ~2-3 hours
  - Quality scoring (500 sample): ~15 minutes
- **Export (Phase 5):** ~1-2 minutes

### References
- [NVIDIA NeMo Data Designer Documentation](https://nvidia-nemo.github.io/DataDesigner/latest/)
- [Data Designer GitHub](https://github.com/NVIDIA-NeMo/DataDesigner)
- [Data Designer on PyPI](https://pypi.org/project/data-designer/)
- [NVIDIA Build API](https://build.nvidia.com)

---